# Chuẩn bị dữ liệu và huấn luyện XGBoost F14 trên Kaggle

**Đầu vào:** CSV thô traffic_flow_dataset_with_paths (2).csv có 29 cột.
**Đầu ra:** hai bộ dữ liệu F14 (identity, robust_log), model XGBoost F14
huấn luyện bằng identity, metric validation, ngưỡng điểm và hồ sơ truy vết.

Mỗi dòng CSV thô là một snapshot; các snapshot của cùng flow_instance_id
được ghép thành **một mẫu học máy**. Model nhận đủ 14 features ở early;
byte tại removal dùng tạo nhãn theo policy.

Notebook dành cho thử nghiệm offline với nhãn **proxy** theo removal;
cutoff thật và completion ứng dụng vẫn cần xác minh độc lập.
Phần 15–20 huấn luyện XGBoost F14 trên train, theo dõi validation,
lưu model/ngưỡng điểm và chỉ chấm test khi bật công tắc.


## 0. Cần upload file nào? Cách bắt đầu

1. Tạo Kaggle Dataset chứa **CSV thô**
   `traffic_flow_dataset_with_paths (2).csv`, hoặc upload gói sẵn
   `datasets/kaggle_upload/f14_preparation_input.zip` chứa CSV và hướng dẫn.
2. Import file `KAGGLE_EARLY_FLOW_BASELINE_TRAINING.ipynb` làm Notebook;
   gắn Kaggle Dataset vừa tạo bằng chức năng **Add Input**.
3. Chọn Python, CPU. Phần train cần NumPy, scikit-learn và XGBoost ≥2.0
   có sẵn; không cần GPU hoặc script ngoài. Phần 15 kiểm phiên bản;
   notebook không tự cài thư viện.
4. Chạy các cell từ trên xuống bằng **Run All**. Chờ đến thông báo
   `TRAIN_READY = True`, xem báo cáo và tải `F14_TRAIN_READY.zip` ở Output.
   Phần 15–19 tiếp tục train/validation và lưu model; phần 20 mặc định
   bỏ qua test. Lưu version có chạy notebook để giữ kết quả.

**Không cần upload** các folder `upload_2_features14_identity_v1` hay
`upload_2_features14_robust_v1`: notebook tự tạo lại từ raw để nhìn rõ
từng bước. Các file X/y cũ đã xử lý, không phải đầu vào của pipeline này.

`/kaggle/input` chứa dữ liệu chỉ đọc; `/kaggle/working` chứa kết quả.
Mỗi lần Run All tạo một thư mục mới để giữ kết quả lần trước.
Hướng dẫn giao diện chính thức: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks).


## 1. Khai báo cấu hình trước khi xem dữ liệu

Giữ nguyên ngưỡng nhãn, cohort và cách chia tập. **PROFILE=identity**
là cấu hình dùng để huấn luyện XGBoost F14 ở phần 15–20.
Phần chuẩn bị xuất cả identity và robust_log cùng X/y/metadata để lưu trữ.
SHA-256 nhận diện đúng CSV, không phải run ID hay chứng nhận chất lượng.

Thường không cần sửa cấu hình. Nếu Kaggle đổi tên file, nhập đầy đủ
đường dẫn vào RAW_PATH_OVERRIDE. Với dataset mới, cập nhật hash hoặc
chủ động đặt EXPECTED_RAW_SHA256=None rồi kiểm tra lại báo cáo.


In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter
import csv
import hashlib
import importlib
import importlib.util
import json
import math
import os
import sys
import uuid
import zipfile

if sys.version_info < (3, 11):
    raise RuntimeError("Cần Python 3.11 trở lên; hãy chọn môi trường Python hiện hành.")

RAW_FILENAME = "traffic_flow_dataset_with_paths (2).csv"
RAW_PATH_OVERRIDE = globals().get("F14_RAW_PATH", os.environ.get("F14_RAW_PATH"))
INPUT_ROOT = Path(globals().get("F14_INPUT_ROOT", os.environ.get("F14_INPUT_ROOT", "/kaggle/input")))
WORK_ROOT = Path(globals().get("F14_WORK_ROOT", os.environ.get("F14_WORK_ROOT", "/kaggle/working/f14_prepared"))).resolve()
EXPECTED_RAW_SHA256 = globals().get("F14_EXPECTED_SHA256", "b26060e9f274c386681754d1204ace7e3dbb8303458679a3c10be3e47674ad4f")
PROFILE = globals().get("F14_PROFILE", "identity")  # hoặc "robust_log"
if PROFILE not in {"identity", "robust_log"}:
    raise ValueError("PROFILE phải là identity hoặc robust_log")
COHORT = "tcp_internal"
THRESHOLD_BYTES = 65536
TRAIN_FRACTION, VALIDATION_FRACTION = 0.70, 0.15
PROFILES = ("identity", "robust_log")
SPLITS = ("train", "validation", "test")
TRAIN_READY = False
EXECUTION_ID = "execution_" + uuid.uuid4().hex
RUN_ROOT = WORK_ROOT / EXECUTION_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
OUTPUT_DIRS = {profile: RUN_ROOT / f"upload_2_features14_{'robust' if profile == 'robust_log' else 'identity'}_v1" for profile in PROFILES}
ZIP_PATH = RUN_ROOT / "F14_TRAIN_READY.zip"
print("Python:", sys.version.split()[0])
print("Kết quả:", RUN_ROOT)
print("PROFILE được chọn:", PROFILE)
print("execution_id chỉ đánh dấu lần chạy notebook; raw run_id vẫn chưa biết.")


## 2. Nạp bộ xử lý đi kèm notebook

Cell kỹ thuật dưới chứa bản sao nguyên vẹn của các module chuẩn bị
dataset đã kiểm thử trong dự án. Nó tự giải nén vào thư mục kết quả,
kiểm hash rồi import trong namespace riêng. **Không cần upload repo**.
Có thể thu gọn cell này; các bước học máy và kết quả vẫn hiện rõ bên dưới.


In [ ]:
# Snapshot tự chứa, được tạo từ source trong repository. Không tải code từ Internet.
EMBEDDED_SOURCES = {'ml_engine/classifier/prepare_dataset.py': '"""Prepare early-flow features and a versioned removal-policy proxy target.\n\nRun from the repository root with ``python -m ml_engine.classifier.prepare_dataset``.\nOnly the Python standard library is required. This module never imports the collector.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nfrom typing import Any\n\nfrom .data_preparation.records import DatasetError, FEATURE_SETS, FlowPair, load_pairs\nfrom .data_preparation.splitting import chronological_split\nfrom .data_preparation.transforms import FeatureTransform\n\n\nTARGET = "y_forward_removal_v1"\nSPLITS = ("train", "validation", "test")\nINTERNAL_LOCALITIES = {"Intra-Rack", "Intra-Cluster", "Intra-DC"}\nQUALITY_FLAGS = [\n    "early_cutoff_unverified",\n    "packet_counter_correction_unverified",\n    "application_completion_unverified",\n    "producer_commit_unknown",\n    "run_id_unknown",\n    "timezone_unknown",\n    "single_artifact_temporal_split_only",\n]\n\n\ndef cohort_reasons(pair: FlowPair) -> list[str]:\n    """Select the documented TCP/internal/sub10ms prototype, using recorded gates."""\n    early = pair.early\n    reasons = []\n    if early.source != "early_probe_sub10ms":\n        reasons.append("not_sub10ms_source")\n    if early.protocol != 6:\n        reasons.append("not_tcp")\n    if early.locality not in INTERNAL_LOCALITIES:\n        reasons.append("not_internal_locality")\n    if not (early.dur <= 0.010 or sum(early.counters[2:]) <= 10):\n        reasons.append("outside_recorded_10ms_or_10packet_gate")\n    return reasons\n\n\ndef _sha256(path: Path) -> str:\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef _code_provenance() -> dict[str, Any]:\n    """Describe preparation code, never claim this commit produced the raw CSV."""\n    root = Path(__file__).resolve().parents[2]\n    commit = None\n    try:\n        result = subprocess.run(\n            ["git", "-c", f"safe.directory={root.as_posix()}", "rev-parse", "HEAD"],\n            cwd=root, capture_output=True, text=True, check=False, timeout=5,\n        )\n        if result.returncode == 0:\n            commit = result.stdout.strip()\n    except (OSError, subprocess.TimeoutExpired):\n        pass\n    files = [Path(__file__), *sorted((Path(__file__).parent / "data_preparation").glob("*.py"))]\n    return {\n        "checkout_commit": commit,\n        "note": "Checkout commit is not verified raw producer provenance; file hashes identify local preparation code.",\n        "python_version": sys.version.split()[0],\n        "files_sha256": {path.relative_to(root).as_posix(): _sha256(path) for path in files},\n    }\n\n\ndef _write_csv(path: Path, columns: list[str], rows: list[list[Any]]) -> None:\n    with path.open("x", encoding="utf-8", newline="") as stream:\n        writer = csv.writer(stream, lineterminator="\\n")\n        writer.writerow(columns)\n        writer.writerows(rows)\n\n\ndef _write_json(path: Path, value: Any) -> None:\n    with path.open("x", encoding="utf-8", newline="\\n") as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)\n        stream.write("\\n")\n\n\nMETADATA_COLUMNS = [\n    "flow_instance_id", "canonical_group", "split", "raw_early_line", "raw_removal_line",\n    "early_timestamp", "early_epoch_ms", "removal_timestamp", "removal_epoch_ms",\n    "src_ip", "dst_ip", "src_port", "dst_port", "protocol", "src_host", "dst_host",\n    "locality", "early_record_source", "removal_record_source", "early_duration_source",\n    "removal_duration_source", "early_dur", "early_hop_count", "early_path",\n    "early_is_completed", "removal_is_completed", "raw_early_label", "raw_removal_label",\n    "removal_sbytes", "removal_dbytes", TARGET, "target_policy", "quality_status",\n]\n\n\ndef _metadata(pair: FlowPair, split: str, threshold: int) -> list[Any]:\n    early, removal = pair.early.raw, pair.removal.raw\n    return [\n        pair.flow_id, pair.canonical_key, split, pair.early.line_number, pair.removal.line_number,\n        early["timestamp"], pair.early.epoch_ms, removal["timestamp"], pair.removal.epoch_ms,\n        *[early[name] for name in ("src_ip", "dst_ip", "src_port", "dst_port", "protocol", "src_host", "dst_host")],\n        early["locality"], early["record_source"], removal["record_source"], early["duration_source"],\n        removal["duration_source"], early["dur"], early["hop_count"], early["path"],\n        early["is_completed"], removal["is_completed"], early["label"], removal["label"],\n        removal["sbytes"], removal["dbytes"], int(pair.removal.counters[0] >= threshold),\n        f"forward_{threshold}B_removal_v1", "timing_and_completion_pending",\n    ]\n\n\ndef _render_report(manifest: dict[str, Any]) -> str:\n    summary = manifest["summary"]\n    lines = [\n        "# Kết quả chuẩn bị dataset", "",\n        "**Trạng thái: prototype dùng nhãn proxy; chưa huấn luyện model.**", "",\n        f"- Raw: `{manifest[\'source\'][\'path\']}`; SHA-256 `{manifest[\'source\'][\'raw_sha256\']}`.",\n        f"- Đọc {summary[\'raw_rows\']:,} snapshots → {summary[\'paired_flows\']:,} flow instance.",\n        f"- Cohort TCP nội bộ/sub10ms: {summary[\'eligible_flows\']:,}; ngoài cohort: {summary[\'cohort_excluded\']:,}.",\n        f"- Purge qua mốc thời gian/nhóm tuple: {summary[\'purged_boundary\']:,} flow.",\n        f"- Feature {manifest[\'configuration\'][\'feature_set\']}: `{\', \'.join(manifest[\'features\'])}`; transform `{manifest[\'configuration\'][\'profile\']}`.",\n        f"- `{TARGET}=1[removal.sbytes >= {manifest[\'target\'][\'threshold_bytes\']}]`; bỏ qua label raw khi dựng y.",\n        "", "| Tập | Số flow | y=0 | y=1 |", "| --- | ---: | ---: | ---: |",\n    ]\n    for split in SPLITS:\n        stats = summary["splits"][split]\n        lines.append(f"| {split} | {stats[\'rows\']} | {stats[\'negative\']} | {stats[\'positive\']} |")\n    lines += [\n        "", "Mốc split dựa trên timestamp epoch được cung cấp, giữ cùng timestamp/canonical tuple trong một nhóm;",\n        "nhóm đi qua mốc hoặc có removal chưa đến trước mốc kế tiếp bị purge. Không chia theo thứ tự dòng CSV.",\n        "Mọi flow ngoài cohort/purge có lý do và raw line trong `flow_audit.csv`.",\n        "", "`X_*.csv` chỉ chứa early features; `y_*.csv` chỉ chứa target; `metadata_*.csv` cùng thứ tự dòng.",\n        "Metadata chứa thông tin tương lai để audit, tuyệt đối không đưa toàn bảng này vào X.",\n        "`transform.json` được fit chỉ trên train và có thể nạp lại cho inference.",\n        "", "Cutoff thực, packet counters và completion ứng dụng chưa được xác minh; tên sub10ms/is_completed",\n        "không chứng minh các điều này. Run ID, múi giờ và producer commit vẫn chưa biết; hash chỉ định danh tệp.",\n        "Split nội bộ một tệp chưa kiểm tra khả năng suy rộng sang run/topology khác.",\n        "Không clip/impute/cân bằng lớp, không tính lại rate từ dur làm tròn; CSV gốc được giữ nguyên.",\n        "", "Chi tiết cấu hình, kiểm tra và hash artifact: `manifest.json`.", "",\n    ]\n    return "\\n".join(lines)\n\n\ndef prepare_dataset(\n    input_path: Path,\n    output_dir: Path,\n    *,\n    feature_set: str = "B0",\n    profile: str = "identity",\n    threshold_bytes: int = 65536,\n    train_fraction: float = 0.70,\n    validation_fraction: float = 0.15,\n) -> dict[str, Any]:\n    """Validate, pair, select, split, fit train-only transforms and export fresh artifacts."""\n    input_path, output_dir = Path(input_path).resolve(), Path(output_dir).resolve()\n    if output_dir.exists():\n        raise DatasetError(f"Output already exists; choose a new directory: {output_dir}")\n    if feature_set not in FEATURE_SETS:\n        raise DatasetError(f"Unknown feature set: {feature_set}")\n    if isinstance(threshold_bytes, bool) or not isinstance(threshold_bytes, int) or threshold_bytes <= 0:\n        raise DatasetError("threshold_bytes must be a positive integer")\n    features = list(FEATURE_SETS[feature_set])\n    transformer = FeatureTransform(features, profile)\n    pairs, source = load_pairs(input_path)\n    exclusions = {pair.flow_id: cohort_reasons(pair) for pair in pairs}\n    eligible = [pair for pair in pairs if not exclusions[pair.flow_id]]\n    assignments, split_info = chronological_split(eligible, train_fraction, validation_fraction)\n    partitions = {split: [pair for pair in eligible if assignments[pair.flow_id] == split] for split in SPLITS}\n    transformer.fit([pair.early.feature_values(features) for pair in partitions["train"]])\n    split_stats = {}\n    for split, members in partitions.items():\n        positive = sum(pair.removal.counters[0] >= threshold_bytes for pair in members)\n        split_stats[split] = {"rows": len(members), "positive": positive, "negative": len(members) - positive}\n        if positive == 0 or positive == len(members):\n            raise DatasetError(f"{split} has only one target class; cannot export a classification split")\n    retained = [pair for members in partitions.values() for pair in members]\n    if len({pair.flow_id for pair in retained}) != len(retained):\n        raise DatasetError("Internal error: flow instance overlap between splits")\n    groups = [{pair.canonical_key for pair in partitions[split]} for split in SPLITS]\n    if any(groups[i] & groups[j] for i in range(3) for j in range(i + 1, 3)):\n        raise DatasetError("Internal error: canonical tuple group overlap between splits")\n    for earlier, later in zip(SPLITS, SPLITS[1:]):\n        if max(pair.removal.epoch_ms for pair in partitions[earlier]) >= min(pair.early.epoch_ms for pair in partitions[later]):\n            raise DatasetError("Internal error: target availability crosses chronological split")\n    all_positive = sum(pair.removal.counters[0] >= threshold_bytes for pair in pairs)\n    reason_counts = Counter(reason for reasons in exclusions.values() for reason in reasons)\n    summary = {\n        "raw_rows": source["row_count"], "paired_flows": len(pairs),\n        "all_pairs_positive": all_positive, "all_pairs_negative": len(pairs) - all_positive,\n        "eligible_flows": len(eligible),\n        "eligible_positive": sum(pair.removal.counters[0] >= threshold_bytes for pair in eligible),\n        "cohort_excluded": len(pairs) - len(eligible),\n        "cohort_exclusion_reason_counts": dict(sorted(reason_counts.items())),\n        "purged_boundary": sum(value == "purged_boundary" for value in assignments.values()),\n        "splits": split_stats,\n    }\n    manifest = {\n        "schema_version": "flow_preparation_v1", "status": "prototype_proxy_only",\n        "source": {**source, "path": str(input_path)},\n        "preparation_code": _code_provenance(),\n        "raw_provenance": {"producer_commit": None, "run_id": None, "timezone": None, "artifact_id": source["raw_sha256"]},\n        "configuration": {\n            "feature_set": feature_set, "profile": profile,\n            "cohort": "early_probe_sub10ms AND protocol=6 AND internal locality AND recorded(dur<=0.01 OR Spkts+Dpkts<=10)",\n            "internal_localities": sorted(INTERNAL_LOCALITIES),\n            "train_fraction_requested": train_fraction, "validation_fraction_requested": validation_fraction,\n        },\n        "features": features,\n        "target": {"name": TARGET, "policy": f"forward_{threshold_bytes}B_removal_v1", "threshold_bytes": threshold_bytes,\n                   "formula": "int(removal.sbytes >= threshold_bytes)", "raw_label_used": False,\n                   "kind": "removal_policy_proxy", "application_completion_verified": False},\n        "quality_flags": QUALITY_FLAGS, "summary": summary, "split": split_info,\n        "checks": {"feature_source": "early_record_only", "flow_instance_disjoint": True,\n                   "canonical_groups_disjoint": True, "target_available_before_next_split": True,\n                   "both_classes_each_split": True, "transform_fit_split": "train",\n                   "raw_sha256_unchanged": True},\n        "artifacts": {},\n    }\n    # Build artifacts off to the side. Reserve a fresh destination only when validation succeeds.\n    output_dir.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix=".flow-prep-", dir=output_dir.parent) as temporary:\n        stage = Path(temporary)\n        for split, members in partitions.items():\n            x_rows = transformer.transform([pair.early.feature_values(features) for pair in members])\n            _write_csv(stage / f"X_{split}.csv", features, x_rows)\n            _write_csv(stage / f"y_{split}.csv", [TARGET], [[int(pair.removal.counters[0] >= threshold_bytes)] for pair in members])\n            _write_csv(stage / f"metadata_{split}.csv", METADATA_COLUMNS, [_metadata(pair, split, threshold_bytes) for pair in members])\n            if not (len(x_rows) == len(members) == split_stats[split]["rows"]):\n                raise DatasetError("Internal error: X/y/metadata alignment")\n        audit_rows = []\n        purge_reasons = split_info.get("purge_reasons", {})\n        for pair in pairs:\n            reasons = exclusions[pair.flow_id]\n            disposition = "excluded_cohort" if reasons else assignments[pair.flow_id]\n            if disposition == "purged_boundary":\n                detail = purge_reasons.get(pair.flow_id, "canonical_group_crosses_split_boundary")\n                reasons = detail if isinstance(detail, list) else [detail]\n            audit_rows.append(_metadata(pair, disposition, threshold_bytes) + [";".join(reasons), len(pair.snapshots)])\n        _write_csv(stage / "flow_audit.csv", METADATA_COLUMNS + ["exclusion_reasons", "snapshot_count"], audit_rows)\n        _write_json(stage / "transform.json", transformer.to_dict())\n        (stage / "report.md").write_text(_render_report(manifest), encoding="utf-8")\n        for path in sorted(stage.iterdir()):\n            row_count = None\n            if path.suffix == ".csv":\n                with path.open(encoding="utf-8", newline="") as stream:\n                    row_count = sum(1 for _ in csv.reader(stream)) - 1\n            manifest["artifacts"][path.name] = {"sha256": _sha256(path), "bytes": path.stat().st_size, "data_rows": row_count}\n        if _sha256(input_path) != source["raw_sha256"]:\n            raise DatasetError("Raw file changed during preparation; no dataset exported")\n        _write_json(stage / "manifest.json", manifest)\n        output_dir.mkdir(exist_ok=False)\n        for path in stage.iterdir():\n            path.rename(output_dir / path.name)\n    return manifest\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Prepare early TCP flow features with a removal-policy proxy label; does not train a model.")\n    parser.add_argument("--input", required=True, type=Path, help="One raw 29-column CSV; never modified")\n    parser.add_argument("--output", required=True, type=Path, help="New output directory; existing paths are rejected")\n    parser.add_argument("--feature-set", choices=sorted(FEATURE_SETS), default="B0")\n    parser.add_argument("--profile", choices=("identity", "log1p", "robust_log"), default="identity")\n    parser.add_argument("--threshold-bytes", type=int, default=65536)\n    parser.add_argument("--train-fraction", type=float, default=0.70)\n    parser.add_argument("--validation-fraction", type=float, default=0.15)\n    args = parser.parse_args(argv)\n    try:\n        manifest = prepare_dataset(args.input, args.output, feature_set=args.feature_set, profile=args.profile,\n                                   threshold_bytes=args.threshold_bytes, train_fraction=args.train_fraction,\n                                   validation_fraction=args.validation_fraction)\n    except (OSError, ValueError, csv.Error) as error:\n        print(f"Dataset preparation failed: {error}", file=sys.stderr)\n        return 2\n    print(json.dumps({"output": str(args.output.resolve()), "status": manifest["status"], "summary": manifest["summary"]}, ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'ml_engine/classifier/prepare_dataset_14.py': '"""Separate 14-feature early-flow preparation; existing B0/B1 stays unchanged.\n\nUse ``python -m ml_engine.classifier.prepare_dataset_14 --help`` from repo root.\nThis exports a removal-policy proxy dataset, never trains or imports controller.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter\nimport csv\nimport json\nfrom pathlib import Path\nimport sys\nimport tempfile\nfrom typing import Any\n\nfrom . import prepare_dataset as common\nfrom .data_preparation.records import DatasetError, FlowPair, load_pairs\nfrom .data_preparation.splitting import chronological_split\nfrom .data_preparation_14.features import FEATURE_NAMES, FEATURE_UNITS, FEATURE_FORMULAS, EPSILON, feature_vector\nfrom .data_preparation_14.transforms import FeatureTransform14\n\n\nCOHORTS = ("tcp_internal", "tcp_udp_internal")\n\n\ndef cohort_reasons_14(pair: FlowPair, cohort: str) -> list[str]:\n    """Reuse B0 scope by default; dual protocol is an explicit extension."""\n    if cohort not in COHORTS:\n        raise DatasetError(f"Unknown cohort: {cohort}")\n    reasons = common.cohort_reasons(pair)\n    if cohort == "tcp_udp_internal":\n        reasons = [reason for reason in reasons if reason != "not_tcp"]\n        if pair.early.protocol not in {6, 17}:\n            reasons.append("not_tcp_or_udp")\n    return reasons\n\n\ndef _provenance() -> dict[str, Any]:\n    result = common._code_provenance()\n    root = Path(__file__).resolve().parents[2]\n    paths = [Path(__file__), *sorted((Path(__file__).parent / "data_preparation_14").glob("*.py"))]\n    result["files_sha256"].update({path.relative_to(root).as_posix(): common._sha256(path) for path in paths})\n    return result\n\n\ndef _schema(profile: str) -> dict[str, Any]:\n    return {\n        "schema_version": "early_flow_14_v1",\n        "feature_order": list(FEATURE_NAMES),\n        "profile": profile,\n        "definitions": [\n            {"name": name, "unit": FEATURE_UNITS[name],\n             "kind": "categorical_code" if name == "protocol" else "numeric",\n             "source": "derived_from_same_early_counters" if name == "pkt_ratio" else "same_early_record",\n             "formula_or_provenance": FEATURE_FORMULAS[name]}\n            for name in FEATURE_NAMES\n        ],\n        "raw_ratio_epsilon": EPSILON,\n        "protocol": {"codes": {"6": "TCP", "17": "UDP"}, "scale": "identity", "one_hot": False},\n        "input_contract": "Untransformed early measurements in feature_order; pkt_ratio derived by feature_vector().",\n        "output_contract": "X files already transformed; do not apply transform.json twice.",\n        "transform_spec": "transform.json",\n        "notes": [\n            "Duration is recorded telemetry, not verified application FCT or cutoff.",\n            "Stored rates and Ratio are preserved in identity; no recalculation from rounded duration or physical clipping.",\n            "Raw pkt_ratio division epsilon and log-ratio pseudocount have different purposes; see transform.json.",\n            "Protocol retains one column, including when constant; categorical treatment is a trainer decision.",\n        ],\n    }\n\n\ndef _report(manifest: dict[str, Any]) -> str:\n    summary = manifest["summary"]\n    lines = [\n        "# Kết quả chuẩn bị dataset riêng cho 14 features", "",\n        "**Trạng thái: prototype với nhãn proxy; chưa train model.**", "",\n        f"- Raw: `{manifest[\'source\'][\'path\']}`; SHA-256 `{manifest[\'source\'][\'raw_sha256\']}`.",\n        f"- {summary[\'raw_rows\']:,} snapshots → {summary[\'paired_flows\']:,} flow instance.",\n        f"- Cohort `{manifest[\'configuration\'][\'cohort\']}`: {summary[\'eligible_flows\']:,} cặp; ngoài cohort {summary[\'cohort_excluded\']:,}; purge {summary[\'purged_boundary\']:,}.",\n        f"- Đủ **14 cột**, đúng thứ tự: `{\', \'.join(FEATURE_NAMES)}`.",\n        f"- Profile `{manifest[\'configuration\'][\'profile\']}`; `pkt_ratio=Spkts/(Dpkts+1e-6)` từ cùng early record.",\n        f"- `{common.TARGET}=int(removal.sbytes >= {manifest[\'target\'][\'threshold_bytes\']})`; không dùng label raw làm target.",\n        "", "| Tập | Số mẫu | y=0 | y=1 |", "| --- | ---: | ---: | ---: |",\n    ]\n    for split in common.SPLITS:\n        stats = summary["splits"][split]\n        lines.append(f"| {split} | {stats[\'rows\']} | {stats[\'negative\']} | {stats[\'positive\']} |")\n    lines += [\n        "", f"Cột hằng trên train (vẫn giữ): `{json.dumps(summary[\'constant_features_train\'], ensure_ascii=False)}`.",\n        f"Trong cohort: {summary[\'early_sbytes_ge_threshold\']} early đã đạt ngưỡng byte; {summary[\'early_dur_gt_10ms\']} early có dur>10ms nhưng đạt gate packet.",\n        f"Có {summary[\'early_dur_equals_0_0099\']} early với dur=0.0099; đây là thống kê giá trị ghi nhận, không chứng minh cơ chế clamp.",\n        "", "Split theo epoch early, giữ ID/canonical tuple trong cùng tập; purge nhóm qua mốc và removal chạm/vượt mốc kế tiếp.",\n        "TCP mặc định dùng cùng cohort/split với B0/B1. TCP+UDP là mở rộng phạm vi, không so trực tiếp metric với B0 TCP để kết luận lợi ích feature.",\n        "", "`X_*.csv` chỉ có early features; `y_*.csv` và `metadata_*.csv` khớp thứ tự dòng. Metadata có thông tin tương lai để audit, không đưa vào X.",\n        "`flow_audit.csv` giữ mọi instance cùng lý do loại; raw line đánh số từ 1, header dòng 1.",\n        "", "`identity` giữ 13 giá trị CSV và thêm pkt_ratio. Profile log dùng log1p cho 8 counts/rates,",\n        "log10((forward+1)/(reverse+1)) cho hai tỷ số; dur/mean/protocol giữ tuyến tính.",\n        "`robust_log` scale 13 cột số theo median/IQR chỉ fit train; protocol giữ mã6/17. IQR=0 dùng mẫu số1.",\n        "X đã biến đổi theo profile; không áp transform lần nữa. Schema/formula/đơn vị: `feature_schema.json`; tham số: `transform.json`.",\n        "", "Duration/rate, cutoff thật, packet correction và completion ứng dụng vẫn chưa xác minh.",\n        "Không clip rate, sửa duration, điền missing, tạo run ID hoặc suy ra múi giờ. Không lấy removal/polling làm X.",\n        "Giữ đủ 14 features là lựa chọn thí nghiệm, chưa chứng minh tốt hơn B0/B1; chọn bằng validation, giữ test cho đánh giá cuối.", "",\n    ]\n    return "\\n".join(lines)\n\n\ndef prepare_dataset_14(\n    input_path: Path,\n    output_dir: Path,\n    *,\n    profile: str = "identity",\n    cohort: str = "tcp_internal",\n    threshold_bytes: int = 65536,\n    train_fraction: float = 0.70,\n    validation_fraction: float = 0.15,\n) -> dict[str, Any]:\n    """Export a fresh 14-feature artifact, reusing validated ID pairs and time splitting."""\n    input_path, output_dir = Path(input_path).resolve(), Path(output_dir).resolve()\n    if output_dir.exists():\n        raise DatasetError(f"Output already exists; choose a new directory: {output_dir}")\n    if cohort not in COHORTS:\n        raise DatasetError(f"Unknown cohort: {cohort}")\n    if isinstance(threshold_bytes, bool) or not isinstance(threshold_bytes, int) or threshold_bytes <= 0:\n        raise DatasetError("threshold_bytes must be a positive integer")\n    transformer = FeatureTransform14(profile)\n    pairs, source = load_pairs(input_path)\n    exclusions = {pair.flow_id: cohort_reasons_14(pair, cohort) for pair in pairs}\n    eligible = [pair for pair in pairs if not exclusions[pair.flow_id]]\n    assignments, split_info = chronological_split(eligible, train_fraction, validation_fraction)\n    partitions = {split: [pair for pair in eligible if assignments[pair.flow_id] == split] for split in common.SPLITS}\n    vectors = {pair.flow_id: feature_vector(pair.early) for pair in eligible}\n    train_rows = [vectors[pair.flow_id] for pair in partitions["train"]]\n    transformer.fit(train_rows)\n    stats = {}\n    for split, members in partitions.items():\n        positive = sum(pair.removal.counters[0] >= threshold_bytes for pair in members)\n        stats[split] = {"rows": len(members), "positive": positive, "negative": len(members) - positive,\n                        "protocol_counts": dict(sorted(Counter(str(pair.early.protocol) for pair in members).items()))}\n        if positive == 0 or positive == len(members):\n            raise DatasetError(f"{split} has only one target class; cannot export a classification split")\n    retained = [pair for members in partitions.values() for pair in members]\n    if len({pair.flow_id for pair in retained}) != len(retained):\n        raise DatasetError("Internal error: flow instance overlap between splits")\n    groups = [{pair.canonical_key for pair in partitions[split]} for split in common.SPLITS]\n    if any(groups[i] & groups[j] for i in range(3) for j in range(i + 1, 3)):\n        raise DatasetError("Internal error: canonical tuple overlap between splits")\n    for before, after in zip(common.SPLITS, common.SPLITS[1:]):\n        if max(pair.removal.epoch_ms for pair in partitions[before]) >= min(pair.early.epoch_ms for pair in partitions[after]):\n            raise DatasetError("Internal error: target availability crosses a split")\n    constants = {}\n    for index, name in enumerate(FEATURE_NAMES):\n        values = {row[index] for row in train_rows}\n        if len(values) == 1:\n            constants[name] = next(iter(values))\n    reason_counts = Counter(reason for reasons in exclusions.values() for reason in reasons)\n    summary = {\n        "raw_rows": source["row_count"], "paired_flows": len(pairs),\n        "eligible_flows": len(eligible),\n        "eligible_positive": sum(pair.removal.counters[0] >= threshold_bytes for pair in eligible),\n        "cohort_excluded": len(pairs) - len(eligible),\n        "cohort_exclusion_reason_counts": dict(sorted(reason_counts.items())),\n        "purged_boundary": sum(value == "purged_boundary" for value in assignments.values()),\n        "retained_flows": len(retained), "splits": stats, "constant_features_train": constants,\n        "early_sbytes_ge_threshold": sum(pair.early.counters[0] >= threshold_bytes for pair in eligible),\n        "early_dur_gt_10ms": sum(pair.early.dur > 0.010 for pair in eligible),\n        "early_dur_equals_0_0099": sum(pair.early.dur == 0.0099 for pair in eligible),\n        "reverse_bytes_zero": sum(pair.early.counters[1] == 0 for pair in eligible),\n    }\n    manifest = {\n        "schema_version": "flow_preparation_14_v1", "status": "prototype_proxy_only",\n        "source": {**source, "path": str(input_path)}, "preparation_code": _provenance(),\n        "raw_provenance": {"producer_commit": None, "run_id": None, "timezone": None, "artifact_id": source["raw_sha256"]},\n        "configuration": {"feature_set": "F14", "profile": profile, "cohort": cohort,\n                          "protocols": [6] if cohort == "tcp_internal" else [6, 17],\n                          "scope_extension": cohort == "tcp_udp_internal",\n                          "early_source": "early_probe_sub10ms", "internal_localities": sorted(common.INTERNAL_LOCALITIES),\n                          "recorded_gate": "dur<=0.01 OR Spkts+Dpkts<=10",\n                          "train_fraction_requested": train_fraction, "validation_fraction_requested": validation_fraction},\n        "features": list(FEATURE_NAMES), "feature_schema": "feature_schema.json",\n        "target": {"name": common.TARGET, "policy": f"forward_{threshold_bytes}B_removal_v1",\n                   "threshold_bytes": threshold_bytes, "formula": "int(removal.sbytes >= threshold_bytes)",\n                   "raw_label_used": False, "kind": "removal_policy_proxy", "application_completion_verified": False},\n        "quality_flags": [*common.QUALITY_FLAGS, "duration_and_rates_cutoff_unverified",\n                          "14_features_not_empirically_selected", "protocol_retained_even_if_constant"],\n        "summary": summary, "split": split_info,\n        "checks": {"feature_count": 14, "feature_source": "same_early_record_only",\n                   "pkt_ratio_source": "same_early_Spkts_and_Dpkts", "flow_instance_disjoint": True,\n                   "canonical_groups_disjoint": True, "target_available_before_next_split": True,\n                   "both_classes_each_split": True, "transform_fit_split": "train",\n                   "raw_sha256_unchanged": True, "existing_B0_B1_not_modified": True},\n        "artifacts": {},\n    }\n    output_dir.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix=".flow-prep-14-", dir=output_dir.parent) as temporary:\n        stage = Path(temporary)\n        for split, members in partitions.items():\n            rows = transformer.transform([vectors[pair.flow_id] for pair in members])\n            if len(rows) != len(members) or any(len(row) != 14 for row in rows):\n                raise DatasetError("Internal error: invalid 14-feature output shape")\n            common._write_csv(stage / f"X_{split}.csv", list(FEATURE_NAMES), rows)\n            common._write_csv(stage / f"y_{split}.csv", [common.TARGET], [[int(pair.removal.counters[0] >= threshold_bytes)] for pair in members])\n            common._write_csv(stage / f"metadata_{split}.csv", common.METADATA_COLUMNS,\n                              [common._metadata(pair, split, threshold_bytes) for pair in members])\n        audit = []\n        for pair in pairs:\n            reasons = exclusions[pair.flow_id]\n            disposition = "excluded_cohort" if reasons else assignments[pair.flow_id]\n            if disposition == "purged_boundary":\n                reasons = split_info["purge_reasons"][pair.flow_id]\n            audit.append(common._metadata(pair, disposition, threshold_bytes) + [";".join(reasons), len(pair.snapshots)])\n        common._write_csv(stage / "flow_audit.csv", common.METADATA_COLUMNS + ["exclusion_reasons", "snapshot_count"], audit)\n        common._write_json(stage / "transform.json", transformer.to_dict())\n        common._write_json(stage / "feature_schema.json", _schema(profile))\n        (stage / "report.md").write_text(_report(manifest), encoding="utf-8")\n        for path in sorted(stage.iterdir()):\n            count = None\n            if path.suffix == ".csv":\n                with path.open(encoding="utf-8", newline="") as stream:\n                    count = sum(1 for _ in csv.reader(stream)) - 1\n            manifest["artifacts"][path.name] = {"sha256": common._sha256(path), "bytes": path.stat().st_size, "data_rows": count}\n        if common._sha256(input_path) != source["raw_sha256"]:\n            raise DatasetError("Raw changed during preparation; no dataset exported")\n        common._write_json(stage / "manifest.json", manifest)\n        output_dir.mkdir(exist_ok=False)\n        for path in stage.iterdir():\n            path.rename(output_dir / path.name)\n    return manifest\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Prepare a separate 14-feature early-flow dataset; does not train a model.")\n    parser.add_argument("--input", type=Path, required=True, help="Raw 29-column CSV with instance IDs; never changed")\n    parser.add_argument("--output", type=Path, required=True, help="Fresh output directory, no overwrite")\n    parser.add_argument("--profile", choices=("identity", "log1p", "robust_log"), default="identity")\n    parser.add_argument("--cohort", choices=COHORTS, default="tcp_internal", help="TCP default matches B0; TCP+UDP explicitly extends scope")\n    parser.add_argument("--threshold-bytes", type=int, default=65536)\n    parser.add_argument("--train-fraction", type=float, default=0.70)\n    parser.add_argument("--validation-fraction", type=float, default=0.15)\n    args = parser.parse_args(argv)\n    try:\n        manifest = prepare_dataset_14(args.input, args.output, profile=args.profile, cohort=args.cohort,\n                                      threshold_bytes=args.threshold_bytes, train_fraction=args.train_fraction,\n                                      validation_fraction=args.validation_fraction)\n    except (OSError, ValueError, csv.Error) as error:\n        print(f"14-feature preparation failed: {error}", file=sys.stderr)\n        return 2\n    print(json.dumps({"output": str(args.output.resolve()), "status": manifest["status"], "features": manifest["features"],\n                      "summary": manifest["summary"]}, ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'ml_engine/classifier/data_preparation/__init__.py': '"""Offline dataset preparation, independent of controller runtime modules."""\n', 'ml_engine/classifier/data_preparation/records.py': '"""Validate the uploaded 29-column CSV and pair records by flow instance ID.\n\nThis module never changes measurements or reads controller modules. Source names,\nthe ID suffix, and removal flags do not prove the producer, an early cutoff, or\napplication completion. Pairing is scoped to one input artifact, not an inferred\nrun. The raw label is preserved and is deliberately not interpreted as a target.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nfrom dataclasses import dataclass\nfrom datetime import datetime\nfrom decimal import Decimal, InvalidOperation\nimport hashlib\nimport io\nfrom ipaddress import IPv4Address\nimport json\nimport math\nfrom pathlib import Path\nimport re\nfrom collections import Counter, defaultdict\nfrom typing import Sequence\n\n\nEARLY_SOURCES = {"early_probe_sub10ms", "early_probe_extended"}\nREMOVAL_SOURCES = {"flow_removed_push", "flow_removed_push_unilateral"}\nCOUNTERS = ("sbytes", "dbytes", "Spkts", "Dpkts")\nFEATURE_SETS = {\n    "B0": COUNTERS,\n    "B1": COUNTERS + ("smeansz", "dmeansz"),\n}\nCOLUMNS = (\n    "timestamp", "flow_instance_id", "src_ip", "dst_ip", "src_port",\n    "dst_port", "protocol", "dur", "sbytes", "dbytes", "Spkts", "Dpkts",\n    "Sload", "Dload", "smeansz", "dmeansz", "Ratio", "hop_count", "path",\n    "label", "src_host", "dst_host", "locality", "pkt_rate", "byte_rate",\n    "timestamp_epoch_ms", "record_source", "duration_source", "is_completed",\n)\nINTEGER_COLUMNS = COUNTERS + (\n    "src_port", "dst_port", "protocol", "hop_count", "label",\n    "timestamp_epoch_ms", "is_completed",\n)\nFLOAT_COLUMNS = (\n    "dur", "Sload", "Dload", "smeansz", "dmeansz", "Ratio", "pkt_rate",\n    "byte_rate",\n)\nSOURCE_PROVENANCE = {\n    "early_probe_sub10ms": ("hardware_early_probe", 0),\n    "early_probe_extended": ("hardware_early_probe", 0),\n    "polling_snapshot": ("ovs_active_duration", 0),\n    "flow_removed_push": ("hardware_flow_removed", 1),\n    "flow_removed_push_unilateral": ("hardware_flow_removed", 1),\n}\n_ID_PATTERN = re.compile(\n    r"flow_(?P<src_ip>[0-9.]+)_(?P<src_port>[0-9]+)_"\n    r"(?P<dst_ip>[0-9.]+)_(?P<dst_port>[0-9]+)_"\n    r"(?P<protocol>[0-9]+)_(?P<suffix>[0-9]{13})"\n)\n_TIMESTAMP_PATTERN = re.compile(\n    r"[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}"\n    r"(?:\\.[0-9]{1,6})?"\n)\n_CALENDAR_ORIGIN = datetime(1970, 1, 1)\n\n\nclass DatasetError(ValueError):\n    """Input or dataset relationships violate the preparation contract."""\n\n\ndef _integer(value: str, column: str, line_number: int) -> int:\n    try:\n        number = Decimal(value)\n        finite_float = float(number)\n    except (InvalidOperation, ValueError, OverflowError) as exc:\n        raise DatasetError(f"Line {line_number}: {column} is not numeric") from exc\n    if not number.is_finite() or not math.isfinite(finite_float):\n        raise DatasetError(f"Line {line_number}: {column} must be finite")\n    if number != number.to_integral_value():\n        raise DatasetError(f"Line {line_number}: {column} must be integral")\n    return int(number)\n\n\ndef _oriented_key(raw: dict[str, str]) -> tuple[str, int, str, int, int]:\n    return (\n        raw["src_ip"], int(Decimal(raw["src_port"])), raw["dst_ip"],\n        int(Decimal(raw["dst_port"])), int(Decimal(raw["protocol"])),\n    )\n\n\n@dataclass(frozen=True)\nclass Record:\n    """A validated row with its original strings and physical CSV start line."""\n\n    raw: dict[str, str]\n    line_number: int\n    epoch_ms: int\n    flow_id: str\n\n    @property\n    def source(self) -> str:\n        return self.raw["record_source"]\n\n    @property\n    def completed(self) -> int:\n        return int(Decimal(self.raw["is_completed"]))\n\n    @property\n    def protocol(self) -> int:\n        return int(Decimal(self.raw["protocol"]))\n\n    @property\n    def locality(self) -> str:\n        return self.raw["locality"]\n\n    @property\n    def dur(self) -> float:\n        return float(self.raw["dur"])\n\n    @property\n    def canonical_key(self) -> str:\n        """Direction-independent tuple for grouping, never an instance join key."""\n        src_ip, src_port, dst_ip, dst_port, protocol = _oriented_key(self.raw)\n        endpoints = sorted(\n            ((src_ip, src_port), (dst_ip, dst_port)),\n            key=lambda endpoint: (int(IPv4Address(endpoint[0])), endpoint[1]),\n        )\n        return json.dumps([protocol, endpoints], separators=(",", ":"))\n\n    @property\n    def counters(self) -> tuple[int, int, int, int]:\n        return tuple(int(Decimal(self.raw[name])) for name in COUNTERS)\n\n    def feature_values(self, names: Sequence[str]) -> list[float]:\n        """Return only supported early feature fields in the requested order."""\n        invalid = set(names) - set(FEATURE_SETS["B1"])\n        if invalid:\n            raise DatasetError(f"Unsupported feature fields: {sorted(invalid)}")\n        return [float(self.raw[name]) for name in names]\n\n\n@dataclass(frozen=True)\nclass FlowPair:\n    """One early/removal pair plus every row of the ID in chronological order."""\n\n    flow_id: str\n    early: Record\n    removal: Record\n    snapshots: list[Record]\n\n    @property\n    def canonical_key(self) -> str:\n        return self.early.canonical_key\n\n\ndef _validate_row(raw: dict[str, str], line_number: int) -> Record:\n    for column, value in raw.items():\n        if not value.strip():\n            raise DatasetError(f"Line {line_number}: {column} is empty")\n\n    integers = {\n        column: _integer(raw[column], column, line_number)\n        for column in INTEGER_COLUMNS\n    }\n    for column, value in integers.items():\n        if column == "label":\n            if value not in {-1, 0, 1}:\n                raise DatasetError(f"Line {line_number}: label must be -1, 0, or 1")\n        elif value < 0:\n            raise DatasetError(f"Line {line_number}: {column} must be nonnegative")\n    for column in ("src_port", "dst_port"):\n        if integers[column] > 65535:\n            raise DatasetError(f"Line {line_number}: {column} exceeds 65535")\n    if integers["protocol"] > 255:\n        raise DatasetError(f"Line {line_number}: protocol exceeds 255")\n    if integers["is_completed"] not in {0, 1}:\n        raise DatasetError(f"Line {line_number}: is_completed must be 0 or 1")\n\n    numbers: dict[str, float] = {}\n    for column in FLOAT_COLUMNS:\n        try:\n            value = float(raw[column])\n        except (ValueError, OverflowError) as exc:\n            raise DatasetError(f"Line {line_number}: {column} is not numeric") from exc\n        if not math.isfinite(value) or value < 0:\n            raise DatasetError(f"Line {line_number}: {column} must be finite and nonnegative")\n        numbers[column] = value\n\n    for column in ("src_ip", "dst_ip"):\n        try:\n            IPv4Address(raw[column])\n        except ValueError as exc:\n            raise DatasetError(f"Line {line_number}: {column} must be IPv4") from exc\n\n    flow_id = raw["flow_instance_id"]\n    id_match = _ID_PATTERN.fullmatch(flow_id)\n    if id_match is None:\n        raise DatasetError(f"Line {line_number}: invalid flow_instance_id format")\n    id_tuple = (\n        id_match["src_ip"], int(id_match["src_port"]), id_match["dst_ip"],\n        int(id_match["dst_port"]), int(id_match["protocol"]),\n    )\n    if id_tuple != _oriented_key(raw):\n        raise DatasetError(f"Line {line_number}: flow_instance_id does not match row tuple")\n\n    # Deliberately do not call timestamp() or assign a timezone: the source\n    # timezone is unknown. This only compares the two supplied calendar axes.\n    if _TIMESTAMP_PATTERN.fullmatch(raw["timestamp"]) is None:\n        raise DatasetError(f"Line {line_number}: timestamp must be a naive calendar timestamp")\n    try:\n        timestamp = datetime.fromisoformat(raw["timestamp"])\n        elapsed = timestamp - _CALENDAR_ORIGIN\n    except ValueError as exc:\n        raise DatasetError(f"Line {line_number}: invalid timestamp") from exc\n    text_microseconds = (\n        (elapsed.days * 86400 + elapsed.seconds) * 1_000_000 + elapsed.microseconds\n    )\n    if abs(text_microseconds - integers["timestamp_epoch_ms"] * 1000) > 1000:\n        raise DatasetError(f"Line {line_number}: timestamp and epoch differ by more than 1 ms")\n\n    source = raw["record_source"]\n    expected_provenance = SOURCE_PROVENANCE.get(source)\n    if expected_provenance is None:\n        raise DatasetError(f"Line {line_number}: unrecognized record_source {source!r}")\n    if (raw["duration_source"], integers["is_completed"]) != expected_provenance:\n        raise DatasetError(f"Line {line_number}: inconsistent provenance/completion for {source}")\n\n    for byte_column, packet_column, mean_column in (\n        ("sbytes", "Spkts", "smeansz"), ("dbytes", "Dpkts", "dmeansz"),\n    ):\n        byte_count = integers[byte_column]\n        packet_count = integers[packet_column]\n        if byte_count > 0 and packet_count == 0:\n            raise DatasetError(f"Line {line_number}: {byte_column} is positive but {packet_column} is zero")\n        expected_mean = byte_count / max(packet_count, 1)\n        if abs(numbers[mean_column] - expected_mean) > 0.0051:\n            raise DatasetError(f"Line {line_number}: {mean_column} does not match counters within rounding tolerance")\n\n    return Record(raw, line_number, integers["timestamp_epoch_ms"], flow_id)\n\n\ndef _record_order(record: Record) -> tuple[int, datetime, int]:\n    return record.epoch_ms, datetime.fromisoformat(record.raw["timestamp"]), record.line_number\n\n\ndef load_pairs(path: Path) -> tuple[list[FlowPair], dict]:\n    """Validate a single CSV artifact fully, then return deterministic ID pairs.\n\n    Missing/extra/duplicate columns, invalid rows, incomplete pairs, duplicate\n    lifecycle events, counter decreases, or records after removal fail the whole\n    artifact. No rows are dropped and the source file is never written.\n    """\n    path = Path(path)\n    try:\n        payload = path.read_bytes()\n    except OSError as exc:\n        raise DatasetError(f"Cannot read input CSV: {exc}") from exc\n    try:\n        text = payload.decode("utf-8-sig")\n    except UnicodeDecodeError as exc:\n        raise DatasetError("Input CSV must be UTF-8") from exc\n\n    reader = csv.reader(io.StringIO(text, newline=""), strict=True)\n    try:\n        columns = next(reader)\n    except StopIteration as exc:\n        raise DatasetError("Input CSV is empty") from exc\n    except csv.Error as exc:\n        raise DatasetError(f"Invalid CSV header: {exc}") from exc\n    duplicates = sorted(name for name, count in Counter(columns).items() if count > 1)\n    missing = sorted(set(COLUMNS) - set(columns))\n    extra = sorted(set(columns) - set(COLUMNS))\n    if duplicates or missing or extra:\n        raise DatasetError(\n            f"Invalid 29-column schema: duplicates={duplicates}, missing={missing}, extra={extra}"\n        )\n\n    rows_by_id: dict[str, list[Record]] = defaultdict(list)\n    source_counts: Counter = Counter()\n    row_count = 0\n    try:\n        while True:\n            line_number = reader.line_num + 1\n            try:\n                values = next(reader)\n            except StopIteration:\n                break\n            if len(values) != len(columns):\n                raise DatasetError(f"Line {line_number}: expected {len(columns)} cells, found {len(values)}")\n            record = _validate_row(dict(zip(columns, values)), line_number)\n            rows_by_id[record.flow_id].append(record)\n            source_counts[record.source] += 1\n            row_count += 1\n    except csv.Error as exc:\n        raise DatasetError(f"Invalid CSV near line {reader.line_num}: {exc}") from exc\n    if not row_count:\n        raise DatasetError("Input CSV contains a header but no data rows")\n\n    pairs: list[FlowPair] = []\n    for flow_id, records in rows_by_id.items():\n        ordered = sorted(records, key=_record_order)\n        early_rows = [record for record in ordered if record.source in EARLY_SOURCES]\n        removal_rows = [record for record in ordered if record.source in REMOVAL_SOURCES]\n        if len(early_rows) != 1 or len(removal_rows) != 1:\n            raise DatasetError(\n                f"Flow {flow_id}: expected one early and one removal; "\n                f"found {len(early_rows)} early and {len(removal_rows)} removal"\n            )\n        early, removal = early_rows[0], removal_rows[0]\n        oriented = _oriented_key(early.raw)\n        if any(_oriented_key(record.raw) != oriented for record in ordered):\n            raise DatasetError(f"Flow {flow_id}: tuple orientation changes within instance")\n        early_time = (early.epoch_ms, datetime.fromisoformat(early.raw["timestamp"]))\n        removal_time = (removal.epoch_ms, datetime.fromisoformat(removal.raw["timestamp"]))\n        if early_time >= removal_time:\n            raise DatasetError(f"Flow {flow_id}: early record must be strictly before removal")\n        if ordered[0] is not early:\n            raise DatasetError(f"Flow {flow_id}: a snapshot precedes the early record")\n        if ordered[-1] is not removal:\n            raise DatasetError(f"Flow {flow_id}: removal must be the terminal record")\n        previous = ordered[0]\n        for record in ordered[1:]:\n            if any(now < before for now, before in zip(record.counters, previous.counters)):\n                raise DatasetError(\n                    f"Flow {flow_id}: counters decrease between lines "\n                    f"{previous.line_number} and {record.line_number}"\n                )\n            previous = record\n        pairs.append(FlowPair(flow_id, early, removal, ordered))\n\n    pairs.sort(key=lambda pair: (pair.early.epoch_ms, pair.flow_id))\n    metadata = {\n        "raw_sha256": hashlib.sha256(payload).hexdigest(),\n        "raw_bytes": len(payload),\n        "row_count": row_count,\n        "flow_count": len(pairs),\n        "columns": columns,\n        "source_counts": dict(sorted(source_counts.items())),\n    }\n    return pairs, metadata\n', 'ml_engine/classifier/data_preparation/splitting.py': '"""Deterministic temporal splits that keep reused canonical tuples together."""\n\nfrom __future__ import annotations\n\nimport math\nfrom collections import defaultdict\nfrom typing import TYPE_CHECKING\n\nif TYPE_CHECKING:\n    from .records import FlowPair\n\n\nSPLIT_NAMES = ("train", "validation", "test")\nPURGED = "purged_boundary"\n\n\ndef _epoch(value: object, context: str) -> int:\n    if isinstance(value, bool) or not isinstance(value, (int, float)):\n        raise ValueError(f"{context} must be an integer epoch in milliseconds")\n    if not math.isfinite(value) or value != int(value):\n        raise ValueError(f"{context} must be a finite integer epoch in milliseconds")\n    return int(value)\n\n\ndef chronological_split(\n    pairs: list[FlowPair],\n    train_fraction: float = 0.7,\n    validation_fraction: float = 0.15,\n) -> tuple[dict[str, str], dict]:\n    """Split by early time, purging whole groups across temporal boundaries.\n\n    Boundaries use *unique* early epoch values, so equal timestamps never split.\n    For n unique epochs, the validation cut index is floor(n*train_fraction),\n    clamped to [1, n-2]. The test cut is floor(n*(train+validation)), clamped\n    to [validation_index+1, n-1]. Fractions therefore approximate proportions\n    of unique timestamps, not proportions of rows, flows, or target classes.\n\n    Every canonical tuple is one conservative split group. A group is purged\n    if its early times occupy different intervals or a removal reaches the\n    next interval\'s start. This does not merge counters of distinct instances.\n    """\n    fractions = (train_fraction, validation_fraction)\n    if any(\n        isinstance(value, bool)\n        or not isinstance(value, (int, float))\n        or not math.isfinite(value)\n        or value <= 0\n        for value in fractions\n    ) or train_fraction + validation_fraction >= 1:\n        raise ValueError("Split fractions must be positive, finite, and sum to less than one")\n\n    by_id: dict[str, FlowPair] = {}\n    groups: dict[str, list[FlowPair]] = defaultdict(list)\n    early_times: dict[str, int] = {}\n    removal_times: dict[str, int] = {}\n    for pair in pairs:\n        if not isinstance(pair.flow_id, str) or not pair.flow_id:\n            raise ValueError("Each pair needs a nonempty flow_id")\n        if pair.flow_id in by_id:\n            raise ValueError(f"Duplicate flow instance in split input: {pair.flow_id}")\n        if not isinstance(pair.canonical_key, str) or not pair.canonical_key:\n            raise ValueError(f"Missing canonical split group for {pair.flow_id}")\n        early_epoch = _epoch(pair.early.epoch_ms, f"{pair.flow_id} early time")\n        removal_epoch = _epoch(pair.removal.epoch_ms, f"{pair.flow_id} removal time")\n        if removal_epoch < early_epoch:\n            raise ValueError(f"Removal precedes early snapshot for {pair.flow_id}")\n        by_id[pair.flow_id] = pair\n        groups[pair.canonical_key].append(pair)\n        early_times[pair.flow_id] = early_epoch\n        removal_times[pair.flow_id] = removal_epoch\n\n    unique_times = sorted(set(early_times.values()))\n    timestamp_count = len(unique_times)\n    if timestamp_count < 3:\n        raise ValueError("At least three distinct early timestamps are required for a temporal split")\n    validation_index = min(max(math.floor(timestamp_count * train_fraction), 1), timestamp_count - 2)\n    test_index = min(\n        max(math.floor(timestamp_count * (train_fraction + validation_fraction)), validation_index + 1),\n        timestamp_count - 1,\n    )\n    validation_start = unique_times[validation_index]\n    test_start = unique_times[test_index]\n\n    def interval(epoch_ms: int) -> str:\n        if epoch_ms < validation_start:\n            return "train"\n        if epoch_ms < test_start:\n            return "validation"\n        return "test"\n\n    assignments: dict[str, str] = {}\n    purge_reasons: dict[str, list[str]] = {}\n    group_counts = dict.fromkeys((*SPLIT_NAMES, PURGED), 0)\n    for canonical_key in sorted(groups):\n        members = sorted(groups[canonical_key], key=lambda pair: pair.flow_id)\n        member_intervals = {interval(early_times[pair.flow_id]) for pair in members}\n        reasons: list[str] = []\n        if len(member_intervals) != 1:\n            reasons.append("boundary_crossing_early")\n        else:\n            assignment = next(iter(member_intervals))\n            next_cutoff = {"train": validation_start, "validation": test_start}.get(assignment)\n            if next_cutoff is not None and any(\n                removal_times[pair.flow_id] >= next_cutoff for pair in members\n            ):\n                reasons.append("boundary_crossing_removal")\n        group_assignment = PURGED if reasons else next(iter(member_intervals))\n        group_counts[group_assignment] += 1\n        for pair in members:\n            assignments[pair.flow_id] = group_assignment\n            if reasons:\n                purge_reasons[pair.flow_id] = list(reasons)\n\n    counts = {name: sum(value == name for value in assignments.values()) for name in (*SPLIT_NAMES, PURGED)}\n    empty_splits = [name for name in SPLIT_NAMES if not counts[name]]\n    if empty_splits:\n        raise ValueError(\n            "Temporal split has empty partitions after boundary purging: "\n            + ", ".join(empty_splits)\n            + ". Supply more independent observations or change the split fractions."\n        )\n\n    observed_ranges = {}\n    for name in SPLIT_NAMES:\n        ids = [flow_id for flow_id, assignment in assignments.items() if assignment == name]\n        observed_ranges[name] = {\n            "min_early_epoch_ms": min(early_times[flow_id] for flow_id in ids),\n            "max_early_epoch_ms": max(early_times[flow_id] for flow_id in ids),\n            "max_removal_epoch_ms": max(removal_times[flow_id] for flow_id in ids),\n        }\n    for previous, following in zip(SPLIT_NAMES, SPLIT_NAMES[1:]):\n        if observed_ranges[previous]["max_removal_epoch_ms"] >= observed_ranges[following]["min_early_epoch_ms"]:\n            raise ValueError(f"Temporal leakage detected between {previous} and {following}")\n\n    summary = {\n        "strategy": "chronological_unique_early_epoch_canonical_group_purge_v1",\n        "requested_fractions": {\n            "train": train_fraction,\n            "validation": validation_fraction,\n            "test": 1.0 - train_fraction - validation_fraction,\n        },\n        "boundary_formula": (\n            "unique sorted early epochs; validation index=clamp(floor(n*train),1,n-2); "\n            "test index=clamp(floor(n*(train+validation)),validation_index+1,n-1); "\n            "indices are zero-based; early < validation, early < test, otherwise test"\n        ),\n        "unique_early_timestamps": timestamp_count,\n        "boundary_indices": {"validation": validation_index, "test": test_index},\n        "thresholds_epoch_ms": {"validation": validation_start, "test": test_start},\n        "input_pairs": len(by_id),\n        "canonical_groups": len(groups),\n        "counts": counts,\n        "group_counts": group_counts,\n        "actual_fractions_of_input": {name: count / len(by_id) for name, count in counts.items()},\n        "purge_reasons": purge_reasons,\n        "observed_ranges": observed_ranges,\n        "temporal_leakage_check_passed": True,\n    }\n    return assignments, summary\n', 'ml_engine/classifier/data_preparation/transforms.py': '"""Serializable train-fitted transforms for the explicitly selected features."""\n\nfrom __future__ import annotations\n\nimport math\n\n\nCOUNT_FEATURES = frozenset(("sbytes", "dbytes", "Spkts", "Dpkts"))\nMEAN_FEATURES = frozenset(("smeansz", "dmeansz"))\nALLOWED_FEATURES = COUNT_FEATURES | MEAN_FEATURES\nPROFILES = frozenset(("identity", "log1p", "robust_log"))\nSCHEMA_VERSION = 1\n\n\ndef _numeric(value: object, context: str, *, positive: bool = False) -> float:\n    if isinstance(value, bool) or not isinstance(value, (int, float)):\n        raise ValueError(f"{context} must be numeric")\n    number = float(value)\n    if not math.isfinite(number) or number < 0 or (positive and number <= 0):\n        domain = "positive" if positive else "nonnegative"\n        raise ValueError(f"{context} must be finite and {domain}")\n    return number\n\n\ndef _quantile(sorted_values: list[float], probability: float) -> float:\n    """Linear interpolation at index (n - 1) * probability."""\n    index = (len(sorted_values) - 1) * probability\n    lower = math.floor(index)\n    upper = math.ceil(index)\n    return sorted_values[lower] + (sorted_values[upper] - sorted_values[lower]) * (index - lower)\n\n\nclass FeatureTransform:\n    """Fit on training rows once, then reuse the saved parameters everywhere.\n\n    ``identity`` preserves all numeric values. ``log1p`` applies log(1+x) only\n    to the four counters. ``robust_log`` first uses that same count transform,\n    leaves mean sizes linear, then scales every selected column by its fitted\n    median and IQR. A zero IQR uses denominator one, without clipping values.\n    Only the six approved B0/B1 feature names are accepted, in caller order.\n    """\n\n    def __init__(self, feature_names: list[str], profile: str = "identity") -> None:\n        if not isinstance(feature_names, list) or not feature_names:\n            raise ValueError("feature_names must be a nonempty list")\n        if any(not isinstance(name, str) or name not in ALLOWED_FEATURES for name in feature_names):\n            raise ValueError("Only selected B0/B1 numeric features are allowed")\n        if len(set(feature_names)) != len(feature_names):\n            raise ValueError("Feature names must not be duplicated")\n        if not isinstance(profile, str) or profile not in PROFILES:\n            raise ValueError(f"Unknown transform profile: {profile!r}")\n        self.feature_names = list(feature_names)\n        self.profile = profile\n        self.medians: list[float] = []\n        self.q25: list[float] = []\n        self.q75: list[float] = []\n        self.scales: list[float] = []\n        self.zero_iqr: list[bool] = []\n        self.num_fit_rows = 0\n\n    def _rows(self, rows: list[list[float]]) -> list[list[float]]:\n        if not isinstance(rows, (list, tuple)):\n            raise ValueError("Rows must be a sequence of numeric feature rows")\n        validated = []\n        for row_index, row in enumerate(rows):\n            if not isinstance(row, (list, tuple)) or len(row) != len(self.feature_names):\n                raise ValueError(f"Feature row {row_index} has an invalid width")\n            validated.append([\n                _numeric(value, f"row {row_index}, feature {name}")\n                for name, value in zip(self.feature_names, row)\n            ])\n        return validated\n\n    def _base(self, row: list[float]) -> list[float]:\n        if self.profile == "identity":\n            return list(row)\n        return [\n            math.log1p(value) if name in COUNT_FEATURES else value\n            for name, value in zip(self.feature_names, row)\n        ]\n\n    def fit(self, rows: list[list[float]]) -> FeatureTransform:\n        """Validate and fit quartiles on the caller\'s training partition only."""\n        validated = self._rows(rows)\n        if not validated:\n            raise ValueError("Cannot fit a transform on an empty training partition")\n        base_rows = [self._base(row) for row in validated]\n        medians, q25, q75, scales, zero_iqr = [], [], [], [], []\n        for column_index in range(len(self.feature_names)):\n            values = sorted(row[column_index] for row in base_rows)\n            lower = _quantile(values, 0.25)\n            upper = _quantile(values, 0.75)\n            iqr = upper - lower\n            medians.append(_quantile(values, 0.5))\n            q25.append(lower)\n            q75.append(upper)\n            scales.append(iqr if iqr > 0 else 1.0)\n            zero_iqr.append(iqr == 0)\n        # Commit fitted parameters only after the complete input is validated.\n        self.medians, self.q25, self.q75 = medians, q25, q75\n        self.scales, self.zero_iqr = scales, zero_iqr\n        self.num_fit_rows = len(validated)\n        return self\n\n    def _require_fitted(self) -> None:\n        if self.num_fit_rows <= 0:\n            raise ValueError("Fit on training rows or load a saved transform before inference")\n\n    def transform(self, rows: list[list[float]]) -> list[list[float]]:\n        """Apply saved parameters without refitting or modifying input rows."""\n        self._require_fitted()\n        base_rows = [self._base(row) for row in self._rows(rows)]\n        if self.profile != "robust_log":\n            return base_rows\n        transformed = [\n            [(value - self.medians[index]) / self.scales[index] for index, value in enumerate(row)]\n            for row in base_rows\n        ]\n        if any(not math.isfinite(value) for row in transformed for value in row):\n            raise ValueError("Transform produced a nonfinite value; check the fitted parameter scale")\n        return transformed\n\n    def to_dict(self) -> dict:\n        """Return a JSON-safe specification, including the exact feature order."""\n        self._require_fitted()\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "feature_order": list(self.feature_names),\n            "profile": self.profile,\n            "quantile_method": "linear_index_(n-1)*p",\n            "count_pretransform": "identity" if self.profile == "identity" else "log1p",\n            "mean_pretransform": "identity",\n            "medians": list(self.medians),\n            "q25": list(self.q25),\n            "q75": list(self.q75),\n            "scales": list(self.scales),\n            "zero_iqr": list(self.zero_iqr),\n            "num_fit_rows": self.num_fit_rows,\n        }\n\n    @classmethod\n    def from_dict(cls, payload: dict) -> FeatureTransform:\n        """Load and validate fitted parameters; never fit inference rows."""\n        if not isinstance(payload, dict):\n            raise ValueError("A transform specification must be an object")\n        version = payload.get("schema_version")\n        if isinstance(version, bool) or not isinstance(version, int) or version != SCHEMA_VERSION:\n            raise ValueError("Unsupported transform schema version")\n        result = cls(payload.get("feature_order"), payload.get("profile"))\n        if payload.get("quantile_method") != "linear_index_(n-1)*p":\n            raise ValueError("Unsupported quantile method")\n        expected_count = "identity" if result.profile == "identity" else "log1p"\n        if payload.get("count_pretransform") != expected_count or payload.get("mean_pretransform") != "identity":\n            raise ValueError("Inconsistent pretransform specification")\n        count = payload.get("num_fit_rows")\n        if isinstance(count, bool) or not isinstance(count, int) or count <= 0:\n            raise ValueError("num_fit_rows must be a positive integer")\n        width = len(result.feature_names)\n        for field in ("medians", "q25", "q75", "scales"):\n            values = payload.get(field)\n            if not isinstance(values, list) or len(values) != width:\n                raise ValueError(f"Invalid {field} parameter width")\n            setattr(result, field, [\n                _numeric(value, f"{field}[{index}]", positive=field == "scales")\n                for index, value in enumerate(values)\n            ])\n        flags = payload.get("zero_iqr")\n        if not isinstance(flags, list) or len(flags) != width or any(type(flag) is not bool for flag in flags):\n            raise ValueError("zero_iqr must contain one boolean per feature")\n        for index, (lower, median, upper, scale, flag) in enumerate(\n            zip(result.q25, result.medians, result.q75, result.scales, flags)\n        ):\n            if not lower <= median <= upper:\n                raise ValueError(f"Unordered fitted quantiles for feature {index}")\n            iqr = upper - lower\n            expected_scale = iqr if iqr > 0 else 1.0\n            if flag != (iqr == 0) or not math.isclose(scale, expected_scale, rel_tol=1e-12, abs_tol=0.0):\n                raise ValueError(f"Inconsistent IQR scale for feature {index}")\n        result.zero_iqr = list(flags)\n        result.num_fit_rows = count\n        return result\n', 'ml_engine/classifier/data_preparation_14/__init__.py': '"""Separate preparation contract for the 14 features proposed in WP05/WP09.\n\nThis package does not change the existing B0/B1 preparation pipeline. It reads\nonly an early record for model inputs; removal records belong to target and\naudit preparation performed by the caller.\n"""\n\nfrom .features import EPSILON, FEATURE_FORMULAS, FEATURE_NAMES, FEATURE_UNITS, LOG_PSEUDOCOUNT, feature_vector\nfrom .transforms import FeatureTransform14\n\n__all__ = [\n    "EPSILON", "FEATURE_FORMULAS", "FEATURE_NAMES", "FEATURE_UNITS",\n    "LOG_PSEUDOCOUNT", "FeatureTransform14", "feature_vector",\n]\n', 'ml_engine/classifier/data_preparation_14/features.py': '"""Extract the complete 14-feature vector from exactly one early CSV record.\n\nThirteen source values are preserved. Only pkt_ratio is newly calculated.\nIn particular, serialized durations can be rounded or clamped, so rates and\nmeans are not recalculated and are not clipped to an assumed link capacity.\n"""\n\nfrom __future__ import annotations\n\nfrom decimal import Decimal, InvalidOperation\nimport math\n\nfrom ..data_preparation.records import COUNTERS, DatasetError, EARLY_SOURCES, Record\n\n\nEPSILON = 1e-6\nLOG_PSEUDOCOUNT = 1.0\nFEATURE_NAMES = (\n    "dur", "sbytes", "dbytes", "Spkts", "Dpkts", "Sload", "Dload",\n    "smeansz", "dmeansz", "Ratio", "pkt_ratio", "byte_rate", "pkt_rate",\n    "protocol",\n)\nFEATURE_UNITS = {\n    "dur": "seconds",\n    "sbytes": "bytes", "dbytes": "bytes",\n    "Spkts": "packets", "Dpkts": "packets",\n    "Sload": "bytes/second", "Dload": "bytes/second",\n    "smeansz": "bytes/packet", "dmeansz": "bytes/packet",\n    "Ratio": "dimensionless", "pkt_ratio": "dimensionless",\n    "byte_rate": "bytes/second", "pkt_rate": "packets/second",\n    "protocol": "IP protocol code (6=TCP, 17=UDP); categorical, not ordinal",\n}\nFEATURE_FORMULAS = {\n    "dur": "early CSV dur preserved; no physical floor or idle-time correction added",\n    "sbytes": "early CSV sbytes preserved",\n    "dbytes": "early CSV dbytes preserved",\n    "Spkts": "early CSV Spkts preserved; collection-time correction is not reversed",\n    "Dpkts": "early CSV Dpkts preserved; collection-time correction is not reversed",\n    "Sload": "early CSV Sload preserved; nominally sbytes/max(dur,epsilon), not recalculated from rounded dur",\n    "Dload": "early CSV Dload preserved; nominally dbytes/max(dur,epsilon), not recalculated from rounded dur",\n    "smeansz": "early CSV smeansz preserved; nominally sbytes/max(Spkts,1)",\n    "dmeansz": "early CSV dmeansz preserved; nominally dbytes/max(Dpkts,1)",\n    "Ratio": "early CSV Ratio preserved; nominally sbytes/(dbytes+epsilon)",\n    "pkt_ratio": "early Spkts/(early Dpkts+epsilon); epsilon=1e-6",\n    "byte_rate": "early CSV byte_rate preserved; nominally (sbytes+dbytes)/max(dur,epsilon), not recalculated from rounded dur",\n    "pkt_rate": "early CSV pkt_rate preserved; nominally (Spkts+Dpkts)/max(dur,epsilon), not recalculated from rounded dur",\n    "protocol": "early CSV protocol preserved as one categorical code column, 6 or 17",\n}\n\n\ndef feature_vector(record: Record) -> list[float]:\n    """Return all 14 fields in FEATURE_NAMES order, exclusively from this row.\n\n    Validation is deliberately repeated at this API boundary so callers cannot\n    pass a terminal or polling record as model input. A validated load_pairs\n    result already satisfies the numeric checks. No final counters, labels,\n    timestamps, addresses, path metadata, or later snapshots are accessed.\n    """\n    if record.source not in EARLY_SOURCES:\n        raise DatasetError(f"Line {record.line_number}: 14-feature inputs require an early record")\n    values: dict[str, float] = {}\n    for name in FEATURE_NAMES:\n        if name == "pkt_ratio":\n            continue\n        try:\n            source = record.raw[name]\n            if isinstance(source, bool):\n                raise ValueError("boolean is not a measurement")\n            number = float(source)\n        except (KeyError, ValueError, TypeError, OverflowError) as exc:\n            raise DatasetError(f"Line {record.line_number}: invalid feature {name}") from exc\n        if not math.isfinite(number) or number < 0:\n            raise DatasetError(f"Line {record.line_number}: feature {name} must be finite and nonnegative")\n        if name in COUNTERS or name == "protocol":\n            try:\n                exact = Decimal(str(source))\n            except InvalidOperation as exc:\n                raise DatasetError(f"Line {record.line_number}: feature {name} must be integral") from exc\n            if exact != exact.to_integral_value():\n                raise DatasetError(f"Line {record.line_number}: feature {name} must be integral")\n            if name == "protocol" and exact not in {Decimal(6), Decimal(17)}:\n                raise DatasetError(f"Line {record.line_number}: protocol must be 6 (TCP) or 17 (UDP)")\n        values[name] = number\n    values["pkt_ratio"] = values["Spkts"] / (values["Dpkts"] + EPSILON)\n    if not math.isfinite(values["pkt_ratio"]):\n        raise DatasetError(f"Line {record.line_number}: pkt_ratio is not finite")\n    return [values[name] for name in FEATURE_NAMES]\n', 'ml_engine/classifier/data_preparation_14/transforms.py': '"""Train-fitted and serializable transforms for the separate 14-feature plan.\n\nIdentity is the tree-model baseline. Optional log1p and robust_log profiles\nreplace the two ratio columns with signed log ratios from the original early\ncounters. Column names and count stay fixed, and the specification records\ntheir changed representation. Protocol codes are always left unchanged.\n"""\n\nfrom __future__ import annotations\n\nimport math\n\nfrom .features import EPSILON, FEATURE_FORMULAS, FEATURE_NAMES, FEATURE_UNITS, LOG_PSEUDOCOUNT\n\n\nSCHEMA_VERSION = 1\nQUANTILE_METHOD = "linear_index_(n-1)*p"\nPROFILES = frozenset(("identity", "log1p", "robust_log"))\nLOG1P_FEATURES = frozenset((\n    "sbytes", "dbytes", "Spkts", "Dpkts", "Sload", "Dload", "byte_rate", "pkt_rate",\n))\nRATIO_FEATURES = frozenset(("Ratio", "pkt_ratio"))\nCOUNT_FEATURES = frozenset(("sbytes", "dbytes", "Spkts", "Dpkts"))\n_INDEX = {name: index for index, name in enumerate(FEATURE_NAMES)}\n\n\ndef _numeric(\n    value: object, context: str, *, nonnegative: bool = True, positive: bool = False,\n) -> float:\n    if isinstance(value, bool) or not isinstance(value, (int, float)):\n        raise ValueError(f"{context} must be numeric")\n    try:\n        number = float(value)\n    except OverflowError as exc:\n        raise ValueError(f"{context} must be finite") from exc\n    if not math.isfinite(number):\n        raise ValueError(f"{context} must be finite")\n    if (positive and number <= 0) or (nonnegative and number < 0):\n        domain = "positive" if positive else "nonnegative"\n        raise ValueError(f"{context} must be {domain}")\n    return number\n\n\ndef _quantile(sorted_values: list[float], probability: float) -> float:\n    """Linear interpolation at index (n - 1) * probability."""\n    index = (len(sorted_values) - 1) * probability\n    lower, upper = math.floor(index), math.ceil(index)\n    return sorted_values[lower] + (sorted_values[upper] - sorted_values[lower]) * (index - lower)\n\n\nclass FeatureTransform14:\n    """Fit only training rows, then reuse the saved state for every partition.\n\n    The caller supplies raw FEATURE_NAMES-order vectors. Identity preserves\n    all values. Both optional profiles log1p the eight counters/rates and use\n    log10(sbytes+1.0)-log10(dbytes+1.0) for Ratio, with the analogous\n    packet expression for pkt_ratio. This avoids division overflow and maps\n    0/0 to zero; a backward-dominant flow can have a negative log ratio.\n    The log pseudocount is 1.0, following WP09, and is independent of the\n    1e-6 denominator epsilon used to derive the raw pkt_ratio feature.\n\n    Robust_log additionally centers and divides the 13 numerical columns by\n    their training median/IQR. Zero IQR uses denominator one. The protocol\n    column remains exactly 6 or 17 in every profile. No clipping or imputation\n    occurs, and neither transform() nor loading a specification refits data.\n    """\n\n    def __init__(self, profile: str = "identity") -> None:\n        if not isinstance(profile, str) or profile not in PROFILES:\n            raise ValueError(f"Unknown 14-feature transform profile: {profile!r}")\n        self.feature_names = list(FEATURE_NAMES)\n        self.profile = profile\n        self.medians: list[float] = []\n        self.q25: list[float] = []\n        self.q75: list[float] = []\n        self.scales: list[float] = []\n        self.zero_iqr: list[bool] = []\n        self.num_fit_rows = 0\n\n    def _rows(self, rows: list[list[float]]) -> list[list[float]]:\n        if not isinstance(rows, (list, tuple)):\n            raise ValueError("Rows must be a sequence of raw 14-feature rows")\n        validated: list[list[float]] = []\n        for row_index, row in enumerate(rows):\n            if not isinstance(row, (list, tuple)) or len(row) != len(FEATURE_NAMES):\n                raise ValueError(f"Feature row {row_index} must contain exactly 14 values")\n            result = [\n                _numeric(value, f"row {row_index}, feature {name}")\n                for name, value in zip(FEATURE_NAMES, row)\n            ]\n            for name in COUNT_FEATURES:\n                if not result[_INDEX[name]].is_integer():\n                    raise ValueError(f"row {row_index}, feature {name} must be integral")\n            if result[_INDEX["protocol"]] not in (6, 17):\n                raise ValueError(f"row {row_index}, protocol must be 6 (TCP) or 17 (UDP)")\n            validated.append(result)\n        return validated\n\n    def _pretransforms(self) -> dict[str, str]:\n        result = {name: "identity" for name in FEATURE_NAMES}\n        if self.profile != "identity":\n            result.update({name: "log1p" for name in LOG1P_FEATURES})\n            result["Ratio"] = "log10(sbytes+log_pseudocount)-log10(dbytes+log_pseudocount)"\n            result["pkt_ratio"] = "log10(Spkts+log_pseudocount)-log10(Dpkts+log_pseudocount)"\n        return result\n\n    def _base(self, row: list[float]) -> list[float]:\n        if self.profile == "identity":\n            return list(row)\n        result = [\n            math.log1p(value) if name in LOG1P_FEATURES else value\n            for name, value in zip(FEATURE_NAMES, row)\n        ]\n        for ratio_name, forward, backward in (\n            ("Ratio", "sbytes", "dbytes"), ("pkt_ratio", "Spkts", "Dpkts"),\n        ):\n            result[_INDEX[ratio_name]] = (\n                math.log10(row[_INDEX[forward]] + LOG_PSEUDOCOUNT)\n                - math.log10(row[_INDEX[backward]] + LOG_PSEUDOCOUNT)\n            )\n        return result\n\n    def fit(self, rows: list[list[float]]) -> FeatureTransform14:\n        validated = self._rows(rows)\n        if not validated:\n            raise ValueError("Cannot fit a transform on an empty training partition")\n        base_rows = [self._base(row) for row in validated]\n        medians, q25, q75, scales, zero_iqr = [], [], [], [], []\n        for column_index in range(len(FEATURE_NAMES)):\n            values = sorted(row[column_index] for row in base_rows)\n            lower, upper = _quantile(values, 0.25), _quantile(values, 0.75)\n            iqr = upper - lower\n            medians.append(_quantile(values, 0.5))\n            q25.append(lower)\n            q75.append(upper)\n            scales.append(iqr if iqr > 0 else 1.0)\n            zero_iqr.append(iqr == 0)\n        # Commit fitted state only after validating every row and parameter.\n        self.medians, self.q25, self.q75 = medians, q25, q75\n        self.scales, self.zero_iqr = scales, zero_iqr\n        self.num_fit_rows = len(validated)\n        return self\n\n    def _require_fitted(self) -> None:\n        if self.num_fit_rows <= 0:\n            raise ValueError("Fit on training rows or load a saved transform before inference")\n\n    def transform(self, rows: list[list[float]]) -> list[list[float]]:\n        self._require_fitted()\n        transformed = [self._base(row) for row in self._rows(rows)]\n        if self.profile == "robust_log":\n            transformed = [\n                [\n                    (value - self.medians[index]) / self.scales[index]\n                    if name != "protocol" else value\n                    for index, (name, value) in enumerate(zip(FEATURE_NAMES, row))\n                ]\n                for row in transformed\n            ]\n        if any(not math.isfinite(value) for row in transformed for value in row):\n            raise ValueError("Transform produced a nonfinite value; check the fitted parameter scale")\n        return transformed\n\n    def _parameters(self) -> dict[str, dict]:\n        pretransforms = self._pretransforms()\n        return {\n            name: {\n                "pretransform": pretransforms[name],\n                "scaled": self.profile == "robust_log" and name != "protocol",\n                "median": self.medians[index],\n                "q25": self.q25[index], "q75": self.q75[index],\n                "scale": self.scales[index], "zero_iqr": self.zero_iqr[index],\n            }\n            for index, name in enumerate(FEATURE_NAMES)\n        }\n\n    def to_dict(self) -> dict:\n        """Return a JSON-safe specification with representation and fit lineage."""\n        self._require_fitted()\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "feature_order": list(FEATURE_NAMES), "profile": self.profile,\n            "epsilon": EPSILON, "log_pseudocount": LOG_PSEUDOCOUNT,\n            "quantile_method": QUANTILE_METHOD,\n            "feature_formulas": dict(FEATURE_FORMULAS),\n            "feature_units": dict(FEATURE_UNITS),\n            "pretransforms": self._pretransforms(),\n            "medians": list(self.medians), "q25": list(self.q25),\n            "q75": list(self.q75), "scales": list(self.scales),\n            "zero_iqr": list(self.zero_iqr), "parameters": self._parameters(),\n            "num_fit_rows": self.num_fit_rows,\n        }\n\n    @classmethod\n    def from_dict(cls, payload: dict) -> FeatureTransform14:\n        """Validate the complete saved state, including signed ratio quantiles."""\n        if not isinstance(payload, dict):\n            raise ValueError("A transform specification must be an object")\n        version = payload.get("schema_version")\n        if type(version) is not int or version != SCHEMA_VERSION:\n            raise ValueError("Unsupported 14-feature transform schema version")\n        if payload.get("feature_order") != list(FEATURE_NAMES):\n            raise ValueError("Transform feature order must match the exact 14-feature contract")\n        result = cls(payload.get("profile"))\n        if _numeric(payload.get("epsilon"), "epsilon", positive=True) != EPSILON:\n            raise ValueError("Unsupported ratio epsilon")\n        if _numeric(payload.get("log_pseudocount"), "log_pseudocount", positive=True) != LOG_PSEUDOCOUNT:\n            raise ValueError("Unsupported log-ratio pseudocount")\n        expected_metadata = {\n            "quantile_method": QUANTILE_METHOD,\n            "feature_formulas": FEATURE_FORMULAS,\n            "feature_units": FEATURE_UNITS,\n            "pretransforms": result._pretransforms(),\n        }\n        for field, expected in expected_metadata.items():\n            if payload.get(field) != expected:\n                raise ValueError(f"Inconsistent {field} specification")\n        count = payload.get("num_fit_rows")\n        if type(count) is not int or count <= 0:\n            raise ValueError("num_fit_rows must be a positive integer")\n        for field in ("medians", "q25", "q75", "scales"):\n            values = payload.get(field)\n            if not isinstance(values, list) or len(values) != len(FEATURE_NAMES):\n                raise ValueError(f"Invalid {field} parameter width")\n            setattr(result, field, [\n                _numeric(\n                    value, f"{field}[{index}]",\n                    nonnegative=not (result.profile != "identity" and name in RATIO_FEATURES and field != "scales"),\n                    positive=field == "scales",\n                )\n                for index, (name, value) in enumerate(zip(FEATURE_NAMES, values))\n            ])\n        flags = payload.get("zero_iqr")\n        if not isinstance(flags, list) or len(flags) != len(FEATURE_NAMES) or any(type(flag) is not bool for flag in flags):\n            raise ValueError("zero_iqr must contain one boolean per feature")\n        for index, (lower, median, upper, scale, flag) in enumerate(\n            zip(result.q25, result.medians, result.q75, result.scales, flags)\n        ):\n            if not lower <= median <= upper:\n                raise ValueError(f"Unordered fitted quantiles for feature {index}")\n            iqr = upper - lower\n            if not math.isfinite(iqr):\n                raise ValueError(f"Nonfinite fitted IQR for feature {index}")\n            expected_scale = iqr if iqr > 0 else 1.0\n            if flag != (iqr == 0) or scale != expected_scale:\n                raise ValueError(f"Inconsistent IQR scale for feature {index}")\n            if FEATURE_NAMES[index] == "protocol" and not (6 <= lower <= median <= upper <= 17):\n                raise ValueError("Protocol fitted quantiles must lie between 6 and 17")\n        result.zero_iqr = list(flags)\n        result.num_fit_rows = count\n        supplied = payload.get("parameters")\n        expected_parameters = result._parameters()\n        if not isinstance(supplied, dict) or set(supplied) != set(FEATURE_NAMES):\n            raise ValueError("parameters must contain the exact 14-feature names")\n        for name, expected in expected_parameters.items():\n            actual = supplied[name]\n            if not isinstance(actual, dict) or set(actual) != set(expected):\n                raise ValueError(f"Invalid parameters for {name}")\n            for field in ("scaled", "zero_iqr"):\n                if type(actual[field]) is not bool or actual[field] != expected[field]:\n                    raise ValueError(f"Inconsistent {field} parameter for {name}")\n            if actual["pretransform"] != expected["pretransform"]:\n                raise ValueError(f"Inconsistent pretransform parameter for {name}")\n            for field in ("median", "q25", "q75", "scale"):\n                value = _numeric(actual[field], f"parameters[{name}][{field}]", nonnegative=False)\n                if value != expected[field]:\n                    raise ValueError(f"Inconsistent {field} parameter for {name}")\n        return result\n'}
EMBEDDED_SOURCE_SHA256 = {'ml_engine/classifier/prepare_dataset.py': '13f8ce7651c613fa6a8ce734fe6912ef3aa2ec3db8e8ce220bf9adf580751dbf', 'ml_engine/classifier/prepare_dataset_14.py': '0b114b6cfb75842f952b8fbb3f70a5c89727fc198373c440ee842330224ccc55', 'ml_engine/classifier/data_preparation/__init__.py': '922a6305bf285d62767b0fe2d03dcc42549defa23215867f8febae585105babb', 'ml_engine/classifier/data_preparation/records.py': '89cc056382399e73ba6763a689333fc10a2b479886aa4c0dfe7d87268db9639f', 'ml_engine/classifier/data_preparation/splitting.py': '8e13b0a1fdbd0404be28fcd189f2736612da5faa96e75b1e9a7e605a3dfc9c78', 'ml_engine/classifier/data_preparation/transforms.py': '49872ed709d7d25402df9b7d851627c3760d10d1a8480511bcd4d17fcf40de1d', 'ml_engine/classifier/data_preparation_14/__init__.py': 'd3a4b0f2e67394d0bf0e2dfbba063683ad5067656d0cc1783e7baf5a703ea4b4', 'ml_engine/classifier/data_preparation_14/features.py': 'e1ea5b5b823f8ca1fbe22cb5af043613d9b69a2cff43ff8bc94e3a84752756d8', 'ml_engine/classifier/data_preparation_14/transforms.py': '730685ad2b55b502ecda0b824ae682a1ba292b74eb05b783a1c9215306e0dbc9'}
SOURCE_ROOT = RUN_ROOT / "pipeline_source"
for relative_name, source_text in EMBEDDED_SOURCES.items():
    payload = source_text.encode("utf-8")
    if hashlib.sha256(payload).hexdigest() != EMBEDDED_SOURCE_SHA256[relative_name]:
        raise RuntimeError(f"Snapshot source bị thay đổi: {relative_name}")
    target = SOURCE_ROOT / relative_name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
for relative_name in ("ml_engine/__init__.py", "ml_engine/classifier/__init__.py"):
    (SOURCE_ROOT / relative_name).write_text("", encoding="utf-8")
SNAPSHOT_PACKAGE = "_kltn_f14_" + EXECUTION_ID
spec = importlib.util.spec_from_file_location(
    SNAPSHOT_PACKAGE, SOURCE_ROOT / "ml_engine/__init__.py",
    submodule_search_locations=[str(SOURCE_ROOT / "ml_engine")],
)
package = importlib.util.module_from_spec(spec)
sys.modules[SNAPSHOT_PACKAGE] = package
spec.loader.exec_module(package)
records_module = importlib.import_module(SNAPSHOT_PACKAGE + ".classifier.data_preparation.records")
splitting_module = importlib.import_module(SNAPSHOT_PACKAGE + ".classifier.data_preparation.splitting")
features_module = importlib.import_module(SNAPSHOT_PACKAGE + ".classifier.data_preparation_14.features")
transforms_module = importlib.import_module(SNAPSHOT_PACKAGE + ".classifier.data_preparation_14.transforms")
preparation_module = importlib.import_module(SNAPSHOT_PACKAGE + ".classifier.prepare_dataset_14")
load_pairs = records_module.load_pairs
chronological_split = splitting_module.chronological_split
feature_vector = features_module.feature_vector
FeatureTransform14 = transforms_module.FeatureTransform14
cohort_reasons_14 = preparation_module.cohort_reasons_14
prepare_dataset_14 = preparation_module.prepare_dataset_14
FEATURE_ORDER = list(features_module.FEATURE_NAMES)
print("Đã nạp", len(EMBEDDED_SOURCES), "module xử lý độc lập.")


## 3. Tìm CSV, kiểm tra đúng phiên bản và cấu trúc

Tìm đúng tên raw trong Input; nếu có nhiều bản, dừng và yêu cầu chọn
`RAW_PATH_OVERRIDE` để tránh đọc nhầm. Nếu Kaggle giữ nguyên ZIP đầu vào,
notebook đọc duy nhất CSV bên trong gói `f14_preparation_input*.zip`;
không giải nén tùy ý hoặc ghi vào Input. Override cũng nhận đường dẫn ZIP.
Kiểm header trước khi xử lý.
Với CSV hiện tại, dự kiến **29 cột** và SHA-256 khớp bản đã khảo sát.


In [ ]:
def csv_from_input_zip(archive_path):
    allowed_names = {RAW_FILENAME, "INPUT_MANIFEST.json", "INPUT_README.txt", "README.md"}
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        names = [member.filename for member in members]
        if len(names) != len(set(names)):
            raise ValueError("ZIP có tên member trùng; cần một bản CSV rõ ràng")
        for member in members:
            name = member.filename
            member_path = PurePosixPath(name)
            if (member.is_dir() or member_path.is_absolute() or ".." in member_path.parts
                    or "\\" in name or name not in allowed_names):
                raise ValueError(f"ZIP có member không hợp lệ: {name!r}")
        if names.count(RAW_FILENAME) != 1:
            raise ValueError("ZIP phải chứa đúng một CSV thô với tên đã quy định")
        payload = archive.read(RAW_FILENAME)
    copied_input = RUN_ROOT / "input_copy" / RAW_FILENAME
    copied_input.parent.mkdir(exist_ok=True)
    with copied_input.open("xb") as stream:
        stream.write(payload)
    return copied_input

if RAW_PATH_OVERRIDE:
    selected_input = Path(RAW_PATH_OVERRIDE).expanduser().resolve()
    if not selected_input.is_file():
        raise FileNotFoundError(f"Không tìm thấy RAW_PATH_OVERRIDE: {selected_input}")
else:
    candidates = sorted(path.resolve() for path in INPUT_ROOT.rglob(RAW_FILENAME) if path.is_file())
    if len(candidates) > 1:
        raise ValueError(f"Cần đúng một raw CSV; tìm được {len(candidates)}: {[str(path) for path in candidates]}. Hãy đặt RAW_PATH_OVERRIDE.")
    if candidates:
        selected_input = candidates[0]
    else:
        archives = sorted(path.resolve() for path in INPUT_ROOT.rglob("f14_preparation_input*.zip") if path.is_file())
        if len(archives) != 1:
            raise ValueError(f"Cần đúng một raw CSV hoặc một ZIP đầu vào; tìm được {len(archives)} ZIP. Hãy kiểm tra Add Input hoặc đặt RAW_PATH_OVERRIDE.")
        selected_input = archives[0]
RAW_INPUT_CONTAINER = selected_input
RAW_PATH = csv_from_input_zip(selected_input) if selected_input.suffix.lower() == ".zip" else selected_input
raw_sha256_before = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
if EXPECTED_RAW_SHA256 is not None and raw_sha256_before != EXPECTED_RAW_SHA256:
    raise ValueError("SHA-256 khác bản upload (2) đã khảo sát. Chọn đúng CSV hoặc cập nhật EXPECTED_RAW_SHA256 một cách chủ động.")
with RAW_PATH.open(encoding="utf-8-sig", newline="") as stream:
    raw_header = next(csv.reader(stream))
print("Raw:", RAW_PATH)
print("Bytes:", RAW_PATH.stat().st_size)
print("SHA-256:", raw_sha256_before)
print("Số cột:", len(raw_header))
print(", ".join(raw_header))


## 4. Kiểm chất lượng và ghép lifecycle theo flow instance

Validator kiểm đủ schema, giá trị hữu hạn, ô trống, kiểu số, ID khớp
tuple, nguồn bản ghi, mean size và thứ tự counters. Mỗi ID phải có đúng
**một early, một removal**, early trước removal; polling được giữ để audit.
Dòng sai làm dừng toàn bộ, không âm thầm điền số hoặc bỏ dòng.

Không nối theo IP/port đơn thuần: cùng tuple có thể được tái sử dụng.
Không cộng counters của các snapshot vì chúng là thống kê cộng dồn.
Không tính lại rate từ `dur` đã làm tròn, không clip theo băng thông giả định.
Validator không thể tự xác nhận cutoff thực/completion hoặc toàn bộ công
thức rate của producer; lần khảo sát raw có hash trên đã kiểm riêng.


In [ ]:
pairs, source = load_pairs(RAW_PATH)
assert source["raw_sha256"] == raw_sha256_before
print(f"{source['row_count']:,} snapshots → {len(pairs):,} flow instance hợp lệ")
print("Các nguồn bản ghi:")
for name, count in source["source_counts"].items():
    print(f"  {name}: {count:,}")
print("Các nhãn raw:", dict(Counter(record.raw['label'] for pair in pairs for record in pair.snapshots)))


### Nhìn một flow cụ thể để hiểu X và y

Dòng early cho biết quan sát đầu luồng; dòng removal cho biết counters
lúc rule bị xóa khỏi switch. `is_completed=1` là cờ lifecycle của collector,
**chưa khẳng định ứng dụng đã truyền xong**. `label=-1` nghĩa là chưa gán nhãn.
Số dòng dưới đây tính cả header là dòng 1 của raw CSV.


In [ ]:
example_pair = next((pair for pair in pairs if pair.early.line_number == 3), pairs[0])
print("flow_instance_id:", example_pair.flow_id)
for record in example_pair.snapshots:
    print({name: record.raw[name] for name in ("timestamp", "record_source", "sbytes", "dbytes", "Spkts", "Dpkts", "is_completed", "label")}, "raw_line=", record.line_number)
print("X lấy tại early dòng", example_pair.early.line_number,
      "; y lấy từ sbytes removal dòng", example_pair.removal.line_number)


## 5. Chọn đúng phạm vi thí nghiệm (cohort)

Mặc định chỉ giữ **TCP, nội bộ DCN, source `early_probe_sub10ms`** và
điều kiện theo số đã ghi: `dur ≤ 0.01 giây HOẶC Spkts+Dpkts ≤ 10`.
Đây là cùng phạm vi của bộ F14 trước đó. UDP, Extra-DC và extended không
dùng trong thí nghiệm này; lý do loại vẫn xuất trong `flow_audit.csv`.

Vì điều kiện là **HOẶC**, early có duration >10 ms vẫn có thể được giữ
nếu số packet ≤10. Tên sub10ms và duration đã ghi không chứng minh cutoff
trên dây. Việc lọc cũng có thể làm đổi phân phối lớp, cần ghi nhận.


In [ ]:
exclusions = {pair.flow_id: cohort_reasons_14(pair, COHORT) for pair in pairs}
eligible = [pair for pair in pairs if not exclusions[pair.flow_id]]
exclusion_counts = Counter(reason for reasons in exclusions.values() for reason in reasons)
print("Được giữ trước split:", len(eligible))
print("Ngoài cohort:", len(pairs) - len(eligible))
print("Lý do loại (một flow có thể có nhiều lý do):", dict(exclusion_counts))
print("Early có dur>10ms nhưng đạt packet gate:", sum(pair.early.dur > 0.01 for pair in eligible))
if not eligible:
    raise ValueError("Không có flow phù hợp cohort; kiểm tra input và mục tiêu thí nghiệm.")


## 6. Tạo nhãn proxy theo policy đã chốt

`y=1` (elephant proxy) nếu **sbytes chiều đi tại removal ≥65.536 byte**;
ngược lại `y=0` (mice proxy). Không dùng tổng hai chiều và không dùng
raw `label=-1`. Removal chỉ dùng cho y/audit; không dùng làm feature.

Đây là dự đoán policy tại removal từ early, chưa phải ground truth
hoàn tất ứng dụng. Không gọi kết quả sau này là accuracy nhận diện sớm
có cutoff đã xác minh nếu chưa có log/trace bổ sung.


In [ ]:
target_by_id = {pair.flow_id: int(pair.removal.counters[0] >= THRESHOLD_BYTES) for pair in eligible}
TARGET_NAME = "y_forward_removal_v1"
print("Policy:", f"forward_{THRESHOLD_BYTES}B_removal_v1")
print("Công thức: int(removal.sbytes >= THRESHOLD_BYTES)")
print("Ví dụ:", example_pair.removal.counters[0], "byte → y =", int(example_pair.removal.counters[0] >= THRESHOLD_BYTES))
print("Đã tạo nhãn cho", len(target_by_id), "flow; thống kê lớp train ở bước 9.")


## 7. Chia train / validation / test trước mọi bước học thống kê

Train để học; validation để theo dõi dừng sớm và chọn ngưỡng điểm;
test chỉ đánh giá cuối. Chia theo **thời gian early tăng dần**, khoảng 70/15/15 các mốc epoch
khác nhau; không ép chính xác tỷ lệ mẫu hoặc tỷ lệ lớp.

Cùng ID chỉ vào một tập. Các instance tái dùng canonical tuple cũng giữ
chung nhóm để thận trọng, nhưng không gộp counters. **Purge** nghĩa là
bỏ khỏi X/y các nhóm qua mốc hoặc có removal chạm/vượt tập kế tiếp,
để đáp án của quá khứ đã có trước đầu vào tương lai. Vẫn giữ lý do audit.
Đây là split trong một artifact; chưa thay thế kiểm chứng trên nhiều run.


In [ ]:
assignments, split_info = chronological_split(eligible, TRAIN_FRACTION, VALIDATION_FRACTION)
partitions = {split: [pair for pair in eligible if assignments[pair.flow_id] == split] for split in SPLITS}
print("Mốc epoch millisecond:", split_info["thresholds_epoch_ms"])
print("Số flow:", split_info["counts"])
print("Không suy ra múi giờ hoặc thời điểm bắt đầu flow từ các epoch/ID chưa xác minh.")
for split in SPLITS:
    print(split, split_info["observed_ranges"][split])


## 8. Lấy đúng 14 features từ cùng một dòng early

Chiều đi là nguồn→đích; chiều về là đích→nguồn. Giữ 13 số đã lưu trong
CSV; chỉ tính thêm `pkt_ratio`. IP, timestamp, path, `is_completed`,
raw label và mọi counters removal nằm ngoài X.

| Feature | Ý nghĩa / đơn vị |
| --- | --- |
| `dur` | Duration được ghi nhận, giây; chưa phải FCT ứng dụng đã xác minh. |
| `sbytes`, `dbytes` | Byte cộng dồn chiều đi / chiều về. |
| `Spkts`, `Dpkts` | Packet cộng dồn chiều đi / chiều về; có thể đã được collector hiệu chỉnh. |
| `Sload`, `Dload` | Rate byte chiều đi / về, **byte/giây**, giữ số CSV. |
| `smeansz`, `dmeansz` | Kích thước packet trung bình mỗi chiều, byte/packet. |
| `Ratio` | Tỷ số byte đi/về đã lưu; có thể rất lớn khi chiều về bằng 0. |
| `pkt_ratio` | `Spkts/(Dpkts+1e-6)` từ cùng early record. |
| `byte_rate` | Rate tổng byte hai chiều, byte/giây, giữ số CSV. |
| `pkt_rate` | Rate tổng packet hai chiều, packet/giây, giữ số CSV. |
| `protocol` | Mã giao thức: 6=TCP, 17=UDP; giữ một cột theo hợp đồng F14. |

Zero chiều về hợp lệ: không điền packet giả. Không tự sửa duration,
rates, đảo hướng hoặc loại `sbytes` chỉ vì dùng byte removal tạo nhãn.
Điều quyết định leakage ở đây là **thời điểm lấy đầu vào**.


In [ ]:
X_raw = {split: [feature_vector(pair.early) for pair in partitions[split]] for split in SPLITS}
y_by_split = {split: [target_by_id[pair.flow_id] for pair in partitions[split]] for split in SPLITS}
flow_ids_by_split = {split: [pair.flow_id for pair in partitions[split]] for split in SPLITS}
print("Thứ tự 14 cột:", FEATURE_ORDER)
print("Mẫu early đầu tiên:", dict(zip(FEATURE_ORDER, X_raw["train"][0])))
for split in SPLITS:
    if len(X_raw[split]) != len(y_by_split[split]) or any(len(row) != 14 for row in X_raw[split]):
        raise ValueError("X/y sai kích thước hoặc mất alignment")
    print(f"{split}: X=({len(X_raw[split])}, 14), y=({len(y_by_split[split])},)")


## 9. Khảo sát riêng train và ghi nhận mất cân bằng lớp

Chỉ dùng train để xem phân phối, cột hằng và gợi ý class weight. Không
quyết định cắt ngoại lệ hoặc chọn feature bằng test. `protocol=6` hằng
trong cohort TCP nhưng vẫn giữ để đúng bộ F14.

Không SMOTE/nhân bản mẫu mặc định. Nếu cần, class weight cân bằng là
`N_train/(2*N_lớp_train)`; đây chỉ là giá trị gợi ý cho giai đoạn train,
phải đánh giá lựa chọn đó bằng validation. Không sửa y hoặc cân bằng test.


In [ ]:
train_class_counts = Counter(y_by_split["train"])
if set(train_class_counts) != {0, 1}:
    raise ValueError("Train cần có cả hai lớp cho bài toán phân loại này")
balanced_class_weight_train = {label: len(y_by_split["train"]) / (2 * train_class_counts[label]) for label in (0, 1)}
constant_features_train = {}
print("Lớp trên train:", dict(train_class_counts))
print("Class weight gợi ý, chỉ từ train:", balanced_class_weight_train)
print("Feature | min | median | max | số zero (chỉ train)")
for column, name in enumerate(FEATURE_ORDER):
    values = sorted(row[column] for row in X_raw["train"])
    middle = (values[(len(values) - 1) // 2] + values[len(values) // 2]) / 2
    print(f"{name:10s} | {values[0]:.6g} | {middle:.6g} | {values[-1]:.6g} | {values.count(0)}")
    if values[0] == values[-1]:
        constant_features_train[name] = values[0]
print("Cột hằng vẫn giữ:", constant_features_train)


## 10. Fit transform chỉ trên train; áp lại nguyên trạng cho validation/test

Hai profile biểu diễn **cùng dữ liệu**, không nối chúng để tăng số mẫu:

- **identity:** giữ số gốc; dùng cho XGBoost F14 ở phần huấn luyện.
- **robust_log:** `log1p` cho 8 counts/rates; hai tỷ số chuyển thành
  `log10((đi+1)/(về+1))`; sau đó 13 cột số được
  `(z−median_train)/IQR_train`, `IQR=Q75−Q25`. IQR=0 dùng mẫu số 1.
  Duration/mean không log; `protocol` giữ mã 6/17, không scale.

Log giảm chênh lệch bậc độ lớn; median/IQR ít nhạy với đuôi dài hơn
mean/std. Phần huấn luyện bên dưới sử dụng identity; robust_log được
xuất cùng bộ hồ sơ chuẩn bị dữ liệu.
**Pseudocount 1** của log-ratio khác **epsilon 1e-6** của `pkt_ratio` gốc.
Số âm sau biến đổi có thể hợp lệ. Không fit lại trên validation/test.


In [ ]:
transformers = {}
X_by_profile = {}
for profile in PROFILES:
    transformer = FeatureTransform14(profile).fit(X_raw["train"])
    transformers[profile] = transformer
    X_by_profile[profile] = {split: transformer.transform(X_raw[split]) for split in SPLITS}
    assert transformer.num_fit_rows == len(X_raw["train"])
    print(profile, "fit_rows=", transformer.num_fit_rows,
          "mẫu đầu=", dict(zip(FEATURE_ORDER, X_by_profile[profile]["train"][0])))
print("Validation/test chỉ gọi transform(); không có fit trên hai tập này.")


## 11. Kiểm tra rò rỉ và hợp đồng dữ liệu trước khi xuất

Kiểm ID/canonical group không giao nhau; removal của tập trước phải sớm
hơn early của tập sau. Kiểm đúng 14 cột, X/y khớp hàng, finite, đủ hai lớp.
Hàm tạo feature từ chối polling/removal. Những kiểm tra này xác minh
pipeline phần mềm; không tự xác minh phép đo cutoff/completion của producer.


In [ ]:
ids = {split: set(flow_ids_by_split[split]) for split in SPLITS}
canonical = {split: {pair.canonical_key for pair in partitions[split]} for split in SPLITS}
for before_index, before in enumerate(SPLITS):
    for after in SPLITS[before_index + 1:]:
        assert not (ids[before] & ids[after]), "Trùng flow ID giữa các tập"
        assert not (canonical[before] & canonical[after]), "Trùng canonical group giữa các tập"
for before, after in zip(SPLITS, SPLITS[1:]):
    assert max(pair.removal.epoch_ms for pair in partitions[before]) < min(pair.early.epoch_ms for pair in partitions[after])
for split in SPLITS:
    assert set(y_by_split[split]) == {0, 1}, f"{split} chỉ có một lớp"
    for pair, row, target in zip(partitions[split], X_raw[split], y_by_split[split]):
        assert row == feature_vector(pair.early)
        assert target == int(pair.removal.counters[0] >= THRESHOLD_BYTES)
    for profile in PROFILES:
        rows = X_by_profile[profile][split]
        assert len(rows) == len(y_by_split[split])
        assert all(len(row) == 14 and all(math.isfinite(value) for value in row) for row in rows)
try:
    feature_vector(partitions["train"][0].removal)
except ValueError:
    pass
else:
    raise AssertionError("Feature extractor đã chấp nhận removal làm X")
print("PASS: early-only, đúng X/y, finite, không trùng ID/group, thời gian không chồng và đủ hai lớp.")


## 12. Xuất artifacts có thể dùng lại và kiểm tra ngược

Hàm xuất dùng cùng validator, cohort, split, feature extractor và
transform vừa học ở trên; nó tính lại để tạo một bộ hoàn chỉnh nguyên vẹn.
Sau xuất, notebook đọc lại CSV/JSON, so với kết quả từng bước và kiểm hash.
Thư mục đích đã tồn tại sẽ bị từ chối; không ghi đè raw hoặc artifact cũ.

| File trong mỗi profile | Dùng làm gì? |
| --- | --- |
| `X_train.csv`, `y_train.csv` | Đầu vào và đáp án để model học sau này. |
| `X_validation.csv`, `y_validation.csv` | Chọn model, tham số và profile. |
| `X_test.csv`, `y_test.csv` | Đánh giá cuối sau khi chốt lựa chọn. |
| `metadata_*.csv` | Truy vết từng mẫu, cùng thứ tự hàng với X/y; không đưa toàn bảng vào model. |
| `flow_audit.csv` | Toàn bộ flow, split hoặc lý do bị loại/purge. |
| `transform.json` | Tham số train để xử lý dữ liệu raw mới; X đã xuất không transform lần hai. |
| `feature_schema.json` | Thứ tự, đơn vị, công thức của 14 features. |
| `manifest.json`, `report.md` | Cấu hình/hash/checks và báo cáo kết quả. |


In [ ]:
def read_csv_rows(path):
    with Path(path).open(encoding="utf-8", newline="") as stream:
        reader = csv.reader(stream)
        return next(reader), list(reader)

manifests = {}
for profile in PROFILES:
    output_dir = OUTPUT_DIRS[profile]
    manifest = prepare_dataset_14(
        RAW_PATH, output_dir, profile=profile, cohort=COHORT,
        threshold_bytes=THRESHOLD_BYTES, train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
    )
    manifests[profile] = manifest
    saved_transform = json.loads((output_dir / "transform.json").read_text(encoding="utf-8"))
    assert saved_transform == transformers[profile].to_dict(), "Tham số không khớp fit trên train"
    for split in SPLITS:
        x_columns, x_rows = read_csv_rows(output_dir / f"X_{split}.csv")
        y_columns, y_rows = read_csv_rows(output_dir / f"y_{split}.csv")
        metadata_columns, metadata_rows = read_csv_rows(output_dir / f"metadata_{split}.csv")
        assert x_columns == FEATURE_ORDER and y_columns == [TARGET_NAME]
        assert [[float(value) for value in row] for row in x_rows] == X_by_profile[profile][split]
        assert [int(row[0]) for row in y_rows] == y_by_split[split]
        assert [row[metadata_columns.index("flow_instance_id")] for row in metadata_rows] == flow_ids_by_split[split]
        assert len(x_rows) == len(y_rows) == len(metadata_rows)
    for name, artifact in manifest["artifacts"].items():
        assert hashlib.sha256((output_dir / name).read_bytes()).hexdigest() == artifact["sha256"], f"Hash sai: {name}"
    assert manifest["source"]["raw_sha256"] == raw_sha256_before
    print(profile, manifest["summary"]["splits"])
assert hashlib.sha256(RAW_PATH.read_bytes()).hexdigest() == raw_sha256_before, "CSV gốc đã thay đổi"
print("PASS: CSV export, thứ tự ID, transform, artifact hashes và raw không đổi.")


## 13. Nạp lại transform cho dữ liệu raw mới (chưa dự đoán model)

Khi triển khai, tạo đủ 14 số **chưa biến đổi** từ cùng early record,
đúng thứ tự schema; nạp `transform.json` đã lưu rồi chỉ gọi `transform`.
Ví dụ này dùng lại một early train để kiểm vòng save→load cho kết quả giống hệt.
Không áp bước này lên `X_*.csv` đã xuất vì các file đó đã biến đổi rồi.


In [ ]:
selected_transform_path = OUTPUT_DIRS[PROFILE] / "transform.json"
reloaded_transform = FeatureTransform14.from_dict(json.loads(selected_transform_path.read_text(encoding="utf-8")))
early_input_untransformed = feature_vector(partitions["train"][0].early)
inference_input_ready = reloaded_transform.transform([early_input_untransformed])[0]
assert inference_input_ready == X_by_profile[PROFILE]["train"][0]
assert reloaded_transform.num_fit_rows == len(X_raw["train"])
print("PASS: transform nạp lại giữ kết quả; không fit lại khi xử lý đầu vào mới.")


## 14. Đóng gói kết quả và bàn giao cho bước train

ZIP chứa hai thư mục profile, source snapshot và hồ sơ chạy notebook;
không nhân bản CSV raw. `execution_id` chỉ nhận diện lần chạy notebook;
`raw_provenance.run_id`, producer và timezone vẫn chưa biết, không tự gán.

Trong Kaggle, xem Output của notebook và tải `F14_TRAIN_READY.zip`
dưới thư mục được in ở cuối. Có thể nạp file X/y bằng pandas/NumPy
trong notebook train tiếp theo; thứ tự dòng của X, y, metadata phải giữ khớp.


In [ ]:
execution_manifest = {
    "schema_version": "kaggle_f14_preparation_v1",
    "execution_id": EXECUTION_ID,
    "raw_sha256": raw_sha256_before,
    "raw_path": str(RAW_PATH),
    "raw_input_container": str(RAW_INPUT_CONTAINER),
    "raw_provenance": {"run_id": None, "producer_commit": None, "timezone": None},
    "selected_profile": PROFILE,
    "feature_order": FEATURE_ORDER,
    "source_snapshot_sha256": EMBEDDED_SOURCE_SHA256,
    "class_weight_balanced_train_only": balanced_class_weight_train,
    "dataset_directories": {profile: path.name for profile, path in OUTPUT_DIRS.items()},
    "split_counts": {split: len(partitions[split]) for split in SPLITS},
    "checks": {"no_model_training": True, "raw_unchanged": True, "export_roundtrip": True,
               "train_only_transform": True, "flow_and_canonical_disjoint": True,
               "target_available_before_next_split": True},
    "status": "prototype_proxy_train_ready",
}
(RUN_ROOT / "notebook_execution.json").write_text(json.dumps(execution_manifest, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
with zipfile.ZipFile(ZIP_PATH, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if (path.is_file() and path != ZIP_PATH and "__pycache__" not in path.parts
                and "input_copy" not in path.relative_to(RUN_ROOT).parts):
            archive.write(path, path.relative_to(RUN_ROOT).as_posix())

# Các biến này đã biến đổi theo PROFILE; sẵn cho code train sau này.
X_train, y_train = X_by_profile[PROFILE]["train"], y_by_split["train"]
X_validation, y_validation = X_by_profile[PROFILE]["validation"], y_by_split["validation"]
X_test, y_test = X_by_profile[PROFILE]["test"], y_by_split["test"]
TRAIN_READY = True
print("TRAIN_READY =", TRAIN_READY)
print("PROFILE =", PROFILE)
print("X_train:", (len(X_train), len(FEATURE_ORDER)), "y_train:", len(y_train))
print("Tải ZIP:", ZIP_PATH)
print("Báo cáo identity:", OUTPUT_DIRS["identity"] / "report.md")
print("Báo cáo robust_log:", OUTPUT_DIRS["robust_log"] / "report.md")


## Kết thúc phần chuẩn bị: chuyển sang phần train bên dưới

Các biến `X_train, y_train, X_validation, y_validation, X_test, y_test`
là list số; mỗi hàng X có 14 cột theo `FEATURE_ORDER`. Chúng đã được xử lý
theo `PROFILE`, không scale lần nữa. `balanced_class_weight_train` chỉ là
gợi ý tùy chọn, chưa được áp vào dữ liệu/model.

Bước tiếp theo: huấn luyện XGBoost F14 bằng identity trên train,
theo dõi validation và chọn ngưỡng điểm, rồi đánh giá test sau khi chốt. Không đưa metadata
có removal bytes vào X; không gộp identity+robust thành thêm mẫu; không
fit scaler hoặc resampling trên cả ba tập. Phần chuẩn bị ở trên **chưa train model**; phần tiếp theo mới huấn luyện.

Với CSV upload (2) và cấu hình mặc định, kỳ vọng: **38.897 snapshots →
15.670 pairs → 9.787 trong cohort → purge 64 → 6.809 train / 1.463
validation / 1.451 test**. Raw có nhãn −1; nhãn được dựng lại bằng policy.
Khác các số này thì xem input/hash/cấu hình/report trước khi dùng.

Sẵn sàng cho code train không đồng nghĩa dataset đã có ground truth
ứng dụng: cutoff thật, packet correction, producer và completion vẫn cần
xác minh riêng. Split một tệp chưa đo khả năng suy rộng sang run khác.


## 15. Kiểm môi trường và cấu hình XGBoost F14

Chạy lần lượt **6 cell code** phần 15–20 sau TRAIN_READY=True.
Model dùng đủ **14 features**, profile identity, nhãn
y=1[removal.sbytes ≥65.536 byte] và split ở trên.
Không thêm scaler hoặc cân bằng lớp. Thiếu thư viện thì dừng;
notebook không tự pip. RUN_FINAL_TEST=False giữ test chưa được chấm điểm.

Đây là thử nghiệm **early → removal proxy**. Điểm tốt chưa xác minh cutoff,
bộ đếm gói hoặc completion ứng dụng. EDA đã xem toàn tệp nên test này chỉ
là đối chứng nội bộ; cần run độc lập để xác nhận quyết định sau EDA.

Tham khảo [XGBoost API và early stopping](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier),
[scikit-learn: chọn ngưỡng](https://scikit-learn.org/stable/modules/classification_threshold.html),
[Average Precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html).


In [ ]:
if not globals().get("TRAIN_READY", False):
    raise RuntimeError("Chạy xong phần chuẩn bị để có TRAIN_READY=True.")
if PROFILE != "identity" or THRESHOLD_BYTES != 65536:
    raise ValueError("Cần PROFILE=identity và THRESHOLD_BYTES=65536.")
try:
    import numpy as np
    import sklearn
    import xgboost
    from xgboost import XGBClassifier
    from sklearn.metrics import (accuracy_score, average_precision_score,
        confusion_matrix, f1_score, precision_score, recall_score)
except ImportError as exc:
    raise RuntimeError("Cần NumPy/scikit-learn/XGBoost có sẵn; không tự cài dependency.") from exc
import re
from datetime import datetime, timezone
from time import perf_counter
release = re.match(r"^(\d+)\.(\d+)", xgboost.__version__)
if release is None or tuple(map(int, release.groups())) < (2, 0):
    raise RuntimeError("Cần XGBoost >=2.0 để dùng device='cpu' và API ở đây.")
PACKAGE_VERSIONS = {"python": sys.version.split()[0], "numpy": np.__version__,
    "scikit_learn": sklearn.__version__, "xgboost": xgboost.__version__}
print("Phiên bản thực tế:", PACKAGE_VERSIONS)
RUN_FINAL_TEST = False
XGB_PARAMS = dict(objective="binary:logistic", tree_method="hist", device="cpu",
    n_estimators=500, max_depth=3, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, random_state=42, n_jobs=2, scale_pos_weight=1.0,
    eval_metric="logloss", early_stopping_rounds=30, verbosity=0)


## 16. Kiểm tra đủ 14 cột và cách chấm điểm

Dùng đủ 14 cột theo FEATURE_ORDER của schema F14. Protocol=6 hằng
trong cohort TCP vẫn được giữ để đúng hợp đồng đầu vào.

**AP** đánh giá thứ tự xếp điểm elephant, không cần ngưỡng 0,5; đây là
average_precision_score, không phải diện tích hình thang dưới đường PR.
Accuracy có thể cao khi đoán toàn mice: xem thêm recall, precision, F1,
FPR và TN/FP/FN/TP. FPR là tỷ lệ mice bị báo nhầm elephant.
Cell này chỉ chuyển train/validation sang NumPy; chưa chấm test.


In [ ]:
EXPECTED_F14 = ["dur", "sbytes", "dbytes", "Spkts", "Dpkts", "Sload", "Dload",
    "smeansz", "dmeansz", "Ratio", "pkt_ratio", "byte_rate", "pkt_rate", "protocol"]
if list(FEATURE_ORDER) != EXPECTED_F14:
    raise ValueError("FEATURE_ORDER khác hợp đồng F14.")
SELECTED_MODEL_NAME = "XGB_F14"
SELECTED_FEATURE_ORDER = list(FEATURE_ORDER)
SELECTED_FEATURE_INDICES = list(range(len(FEATURE_ORDER)))
def checked_split(rows, labels):
    values, targets = np.asarray(rows, dtype=np.float64), np.asarray(labels)
    if (values.ndim != 2 or values.shape[1] != 14 or targets.ndim != 1
            or len(values) != len(targets) or not len(values)
            or not np.isfinite(values).all() or set(np.unique(targets).tolist()) != {0, 1}):
        raise ValueError("X/y cần finite, đúng 14 cột, khớp hàng và đủ hai lớp 0/1.")
    return values, targets.astype(np.int64)
TRAIN_X, TRAIN_Y = checked_split(X_train, y_train)
VALIDATION_X, VALIDATION_Y = checked_split(X_validation, y_validation)
PREPARATION_DIR = Path(OUTPUT_DIRS["identity"])
PREPARATION_MANIFEST_BYTES = (PREPARATION_DIR / "manifest.json").read_bytes()
PREPARATION_MANIFEST = json.loads(PREPARATION_MANIFEST_BYTES)
if (PREPARATION_MANIFEST["configuration"]["profile"] != "identity"
        or PREPARATION_MANIFEST["target"]["threshold_bytes"] != 65536
        or PREPARATION_MANIFEST["features"] != list(FEATURE_ORDER)):
    raise ValueError("Manifest nguồn khác profile/target/thứ tự cột.")
if (len(TRAIN_X) != PREPARATION_MANIFEST["summary"]["splits"]["train"]["rows"]
        or len(VALIDATION_X) != PREPARATION_MANIFEST["summary"]["splits"]["validation"]["rows"]):
    raise ValueError("Số dòng khác manifest nguồn.")
def positive_scores(model, values):
    col = np.flatnonzero(np.asarray(model.classes_) == 1)
    if len(col) != 1:
        raise ValueError("Model thiếu cột xác suất lớp 1.")
    scores = np.asarray(model.predict_proba(values)[:, int(col[0])], dtype=float)
    if not np.isfinite(scores).all():
        raise ValueError("Điểm dự đoán không hữu hạn.")
    return scores
def score_metrics(targets, scores, threshold=0.5):
    pred = (scores >= threshold).astype(np.int64)
    tn, fp, fn, tp = map(int, confusion_matrix(targets, pred, labels=[0, 1]).ravel())
    return {"AP": float(average_precision_score(targets, scores)),
        "accuracy": float(accuracy_score(targets, pred)),
        "precision": float(precision_score(targets, pred, zero_division=0)),
        "recall": float(recall_score(targets, pred, zero_division=0)),
        "F1": float(f1_score(targets, pred, zero_division=0)),
        "FPR": fp / (fp + tn) if fp + tn else 0.0,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp}
print("Train:", TRAIN_X.shape, "Validation:", VALIDATION_X.shape)
print("Thứ tự 14 cột F14:", SELECTED_FEATURE_ORDER)


## 17. Huấn luyện XGBoost F14 và xem validation

Huấn luyện một model XGBoost F14 bằng CPU, tối đa 500 cây; dừng sớm
sau 30 vòng validation logloss không cải thiện.
**eval_set chỉ chứa validation**. Thời gian trong bảng chỉ đo fit,
không gồm chuẩn bị hoặc suy luận. Các metric dùng ngưỡng điểm 0,5.


In [ ]:
f14_model = XGBClassifier(**XGB_PARAMS)
started = perf_counter()
f14_model.fit(
    TRAIN_X, TRAIN_Y,
    eval_set=[(VALIDATION_X, VALIDATION_Y)],
    verbose=False,
)
fit_seconds = perf_counter() - started
validation_scores = positive_scores(f14_model, VALIDATION_X)
validation_row = {
    "model": SELECTED_MODEL_NAME,
    "feature_count": len(SELECTED_FEATURE_ORDER),
    "score_threshold": 0.5,
    "fit_seconds": float(fit_seconds),
    "best_iteration": int(f14_model.best_iteration),
    **score_metrics(VALIDATION_Y, validation_scores),
}
print("Kết quả VALIDATION của XGBoost F14 tại score >=0.5:")
print(json.dumps(validation_row, ensure_ascii=False, allow_nan=False))


## 18. Chọn ngưỡng điểm của F14 trên validation

Với model F14 đã huấn luyện, thử ngưỡng 0,05;0,10;…;0,95 và chọn
F1 cao nhất. Hòa F1 thì ưu tiên FPR thấp hơn, ngưỡng gần 0,5,
rồi ngưỡng cao hơn. Không tối ưu bằng test.
F1 cần đọc cùng số báo nhầm và số bỏ sót trước khi dùng trong hệ thống.

**Hai ngưỡng khác nhau:** 65.536 **byte** tạo nhãn đáp án từ removal;
ngưỡng **điểm dự đoán** quyết định model báo mice hay elephant.
Đổi ngưỡng điểm không đổi nhãn/policy byte.


In [ ]:
THRESHOLD_ROWS = [
    {
        "score_threshold": step / 20.0,
        **score_metrics(VALIDATION_Y, validation_scores, step / 20.0),
    }
    for step in range(1, 20)
]
best_threshold_row = sorted(
    THRESHOLD_ROWS,
    key=lambda row: (
        -row["F1"], row["FPR"],
        abs(row["score_threshold"] - 0.5), -row["score_threshold"],
    ),
)[0]
SELECTED_SCORE_THRESHOLD = float(best_threshold_row["score_threshold"])
print("Model:", SELECTED_MODEL_NAME, "Cột:", SELECTED_FEATURE_ORDER)
print("Nhãn byte:", THRESHOLD_BYTES, "Ngưỡng điểm:", SELECTED_SCORE_THRESHOLD)
print("VALIDATION sau chốt ngưỡng:", best_threshold_row)


## 19. Lưu model F14 và ngưỡng điểm đã chốt

Tạo thư mục mới dưới RUN_ROOT/training bằng UTC+UUID, không ghi đè.
Lưu model JSON, bảng validation, bảng ngưỡng, selection.json và bản sao
manifest/transform/schema nguồn. Selection ghi đủ 14 cột đúng thứ tự,
hai ngưỡng, tham số, best iteration, phiên bản và hash.

Giữ model F14 đã dừng sớm; **không fit lại train+validation**. Suy luận:
tạo đủ F14 theo schema, áp identity transform và giữ nguyên thứ tự cột.
ZIP F14 chỉ có dữ liệu; tải thêm thư mục training để có model/kết quả.
Cờ no_model_training ở notebook_execution.json mô tả riêng phần chuẩn bị;
selection.json ghi nhận lần huấn luyện F14.


In [ ]:
training_parent = Path(RUN_ROOT).resolve() / "training"
training_parent.mkdir(exist_ok=True)
TRAINING_DIR = training_parent / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid.uuid4().hex[:12])
TRAINING_DIR.mkdir(exist_ok=False)
def write_training_csv(path, rows):
    with Path(path).open("x", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
write_training_csv(TRAINING_DIR / "validation_results.csv", [validation_row])
write_training_csv(TRAINING_DIR / "threshold_search.csv", THRESHOLD_ROWS)
for old, new in [("manifest.json", "source_manifest.json"),
                 ("transform.json", "transform.json"),
                 ("feature_schema.json", "feature_schema.json")]:
    payload = (PREPARATION_DIR / old).read_bytes()
    if old == "manifest.json":
        if payload != PREPARATION_MANIFEST_BYTES:
            raise RuntimeError("Manifest nguồn thay đổi.")
    elif hashlib.sha256(payload).hexdigest() != PREPARATION_MANIFEST["artifacts"][old]["sha256"]:
        raise RuntimeError("Artifact nguồn thay đổi: " + old)
    with (TRAINING_DIR / new).open("xb") as stream:
        stream.write(payload)
MODEL_PATH = TRAINING_DIR / "selected_model.json"
f14_model.save_model(str(MODEL_PATH))
MODEL_SHA256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
selection = {
    "schema_version": "early_flow_f14_training_v1",
    "status": "prototype_early_to_removal_proxy",
    "selected_model": SELECTED_MODEL_NAME,
    "selected_feature_order": SELECTED_FEATURE_ORDER,
    "source_feature_order": list(FEATURE_ORDER),
    "selected_feature_indices": SELECTED_FEATURE_INDICES,
    "input_recipe": "raw early F14 -> identity transform -> all 14 columns in saved order",
    "profile": PROFILE, "target_policy": "forward_65536B_removal_v1",
    "label_threshold_bytes": THRESHOLD_BYTES,
    "score_threshold": SELECTED_SCORE_THRESHOLD,
    "training_design": "single XGB_F14 on train; early stopping on validation logloss",
    "threshold_selection": "validation F1; tie FPR, distance to0.5, higher threshold",
    "threshold_grid": [r["score_threshold"] for r in THRESHOLD_ROWS],
    "validation_at_0_5": validation_row,
    "validation_at_selected_threshold": best_threshold_row,
    "requested_xgb_params": XGB_PARAMS,
    "best_iteration": int(f14_model.best_iteration),
    "packages": PACKAGE_VERSIONS,
    "preparation_manifest_sha256": hashlib.sha256(PREPARATION_MANIFEST_BYTES).hexdigest(),
    "raw_sha256": PREPARATION_MANIFEST["source"]["raw_sha256"],
    "model_sha256": MODEL_SHA256,
    "transform_sha256": hashlib.sha256((TRAINING_DIR / "transform.json").read_bytes()).hexdigest(),
    "validation_table_sha256": hashlib.sha256((TRAINING_DIR / "validation_results.csv").read_bytes()).hexdigest(),
    "train_rows": int(len(TRAIN_X)), "validation_rows": int(len(VALIDATION_X)),
    "refit_train_plus_validation": False,
    "test_used_for_model_or_threshold_selection": False,
    "quality_notes": ["cutoff/counters/application completion unverified",
        "single-artifact split; EDA already viewed whole artifact",
        "independent run needed for confirmatory evaluation",
        "fit_seconds excludes preprocessing and prediction"],
}
SELECTION_PATH = TRAINING_DIR / "selection.json"
with SELECTION_PATH.open("x", encoding="utf-8", newline="\n") as stream:
    json.dump(selection, stream, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False)
    stream.write("\n")
LOCKED_SELECTION_SHA256 = hashlib.sha256(SELECTION_PATH.read_bytes()).hexdigest()
print("Đã chốt selection SHA-256:", LOCKED_SELECTION_SHA256)
print("Thư mục training:", TRAINING_DIR, "Model:", MODEL_PATH)
print("Test chưa được chấm điểm; xem công tắc cuối.")


## 20. Test cuối của F14 — mặc định bỏ qua

RUN_FINAL_TEST=False chỉ in thông báo bỏ qua, không dự đoán/tính metric test.
Sau khi xem validation và chốt ngưỡng điểm, bỏ dấu # ở dòng RUN_FINAL_TEST=True
trong cell dưới rồi chạy **riêng cell này**. Cell đọc lại model F14 và
selection.json, kiểm hash, chấm đúng model/ngưỡng đã chốt và tạo
test_metrics.json. Có kết quả cũ thì từ chối chạy lại.

Không đổi model, bộ cột hoặc ngưỡng theo test. Nếu đổi thiết kế vì đã xem
test, coi đó là dữ liệu phát triển và dùng run độc lập cho đánh giá cuối.
Metric ở tệp hiện tại vẫn chỉ mô tả prototype proxy.


In [ ]:
# Sau khi chốt phần 19, bỏ dấu # dòng dưới để chấm test một lần.
# RUN_FINAL_TEST = True
if not RUN_FINAL_TEST:
    print("BỎ QUA TEST: RUN_FINAL_TEST=False. Chưa tạo test_metrics.json.")
else:
    final_test_path = TRAINING_DIR / "test_metrics.json"
    if final_test_path.exists():
        raise FileExistsError("Đã có test_metrics.json; không chấm lại để chọn model/ngưỡng.")
    if hashlib.sha256(SELECTION_PATH.read_bytes()).hexdigest() != LOCKED_SELECTION_SHA256:
        raise RuntimeError("selection.json khác lựa chọn đã chốt.")
    locked = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
    if hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest() != locked["model_sha256"]:
        raise RuntimeError("Model JSON khác model đã chốt.")
    if list(FEATURE_ORDER) != locked["source_feature_order"]:
        raise RuntimeError("Thứ tự F14 thay đổi sau khi chốt.")
    TEST_X, TEST_Y = checked_split(X_test, y_test)
    locked_model = XGBClassifier(n_jobs=2, device="cpu")
    locked_model.load_model(str(MODEL_PATH))
    test_scores = positive_scores(locked_model, TEST_X[:, locked["selected_feature_indices"]])
    final_test_result = {"split": "test", "selected_model": locked["selected_model"],
        "selected_feature_order": locked["selected_feature_order"],
        "label_threshold_bytes": locked["label_threshold_bytes"],
        "score_threshold": locked["score_threshold"], "rows": int(len(TEST_X)),
        "selection_sha256": LOCKED_SELECTION_SHA256, "model_sha256": locked["model_sha256"],
        "metrics": score_metrics(TEST_Y, test_scores, locked["score_threshold"]),
        "status": "prototype_proxy_single_artifact_test"}
    with final_test_path.open("x", encoding="utf-8", newline="\n") as stream:
        json.dump(final_test_result, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n")
    print(json.dumps(final_test_result, ensure_ascii=False, indent=2, allow_nan=False))
    print("Đã lưu:", final_test_path)
